# ARC Prize 2026 (ARC-AGI-2) — H4: Verified Ensemble + Holistic Judge, pass@2-optimised

Ports the two ARC-Prize-**verified** harness mechanisms to a **locally-served Qwen3.8-27B** (no external APIs ⇒ Kaggle-eligible):

1. **Poetiq code-refinement engine** (54.0% semi-private, verified): 2 experts (8+4 iterations), pool-5 scored feedback, shuffled train order per iteration, sandbox verification, early-exit on full train match (precision ~97%).
2. **Land holistic trace judge** (72.9% semi-private): ONE long-context prompt with ALL candidate traces per test input; council of 3 samples at `reasoning_effort=medium`; 2 pts per first choice / 1 pt per second choice; may **synthesize** a new grid.
3. **pass@2-aware attempt selection**: shape-prior repair hedge + source decorrelation (attempt_2 maximises conditional coverage, not rank).

Budget: ~99K GPU-effective tokens/task worst case, ~60–80K expected; degradation ladder at 60/75/88/93% of wall clock; cumulative `submission.json` write after every task; watchdog finalises 60 s before the deadline (12 h − 600 s buffer pattern).

**Required attachments** (offline rerun — internet is disabled):
- Competition data: `arc-prize-2026-arc-agi-2`
- Model dataset: `saltb0x/qwen3-8-27b-fp8` (25.3 GB FP8; Ada sm_89-safe — do NOT use NVFP4 mirrors)
- vLLM: preinstalled in the current Kaggle image; if missing, attach a vLLM wheelhouse dataset and the notebook installs it `--no-index`.

If GPUs / model / vLLM are unavailable, the notebook still writes a schema-valid fallback submission (identity + modal-colour grids). **Never lose the submission.**

See `HYPOTHESIS.md` in the repo for the full claim, budget arithmetic and ablation design.

In [ ]:
# --- H4 setup: deadline (12h - 600s buffer, lb3389 pattern), paths, package dir ---
import os, sys, time, json

NOTEBOOK_START = time.time()
GLOBAL_END_TIME = NOTEBOOK_START + 12 * 3600 - 600  # 600s submission/write buffer
os.makedirs("/kaggle/working/harness", exist_ok=True)
if "/kaggle/working" not in sys.path:
    sys.path.insert(0, "/kaggle/working")

rerun_mode = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
comp_dir = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
challenges_path = os.path.join(
    comp_dir,
    "arc-agi_test_challenges.json" if rerun_mode else "arc-agi_evaluation_challenges.json",
)
print("rerun_mode:", rerun_mode)
print("challenges:", challenges_path, os.path.exists(challenges_path))
print("global end time in", round(GLOBAL_END_TIME - time.time()), "s")

In [ ]:
%%writefile /kaggle/working/harness/__init__.py
"""H4 harness: Poetiq code-refinement + Land holistic judge, pass@2-optimised.

Portable, dependency-light (numpy only at runtime; scipy/cv2 optional inside the
sandbox). Runs against any OpenAI-compatible endpoint (vLLM) or MockLLM.
"""

from harness.types import (  # noqa: F401
    Candidate,
    Grid,
    GridStat,
    JudgePick,
    LLMResponse,
    RunResult,
)
from harness.config import PipelineConfig  # noqa: F401

__all__ = [
    "Candidate",
    "Grid",
    "GridStat",
    "JudgePick",
    "LLMResponse",
    "RunResult",
    "PipelineConfig",
]


In [ ]:
%%writefile /kaggle/working/harness/types.py
"""Core data types for the H4 harness."""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Optional

Grid = list[list[int]]


@dataclass
class LLMResponse:
    text: str = ""
    prompt_tokens: int = 0
    completion_tokens: int = 0
    error: Optional[str] = None
    duration: float = 0.0

    @property
    def ok(self) -> bool:
        return self.error is None


@dataclass
class RunResult:
    """Result of executing one candidate program on one grid (train or test)."""

    success: bool = False
    grid: Optional[Grid] = None
    raw_output: str = ""
    soft_score: float = 0.0
    error: Optional[str] = None


@dataclass
class Candidate:
    """One hypothesis produced by any channel (code expert iteration or text sample)."""

    source: str  # "expert_a" | "expert_b" | "text" | ...
    iteration: int = 0
    trace: str = ""  # model-visible reasoning/code, fed to the judge (truncated later)
    code: Optional[str] = None
    grids: list[Optional[Grid]] = field(default_factory=list)  # per test input
    verified: bool = False  # all train pairs matched exactly
    soft: float = 0.0  # mean per-train-pair soft score
    feedback: str = ""


@dataclass
class JudgePick:
    """One judge's top-2 grids (Land: last two distinct grids in the response)."""

    grids: list[Grid] = field(default_factory=list)
    error: Optional[str] = None
    prompt_tokens: int = 0
    completion_tokens: int = 0


@dataclass
class ScoreEntry:
    """Aggregated judge scoreboard entry (2 pts first choice, 1 pt second choice)."""

    grid: Grid
    points: int = 0
    origin: str = "candidate"  # "candidate" | "synthesized"
    matched: bool = True
    voters: list[int] = field(default_factory=list)


@dataclass
class GridStat:
    """Per-distinct-grid statistics for one test input, used by attempt selection."""

    grid: Grid
    judge_points: int = 0
    verified_votes: int = 0
    votes: int = 0
    best_soft: float = 0.0
    sources: set[str] = field(default_factory=set)
    synthesized: bool = False

    @property
    def shape(self) -> tuple[int, int]:
        return (len(self.grid), len(self.grid[0]) if self.grid else 0)


In [ ]:
%%writefile /kaggle/working/harness/grids.py
"""Grid utilities: formatting, tolerant CSV parsing (ported from Land's src/grid.py
and src/judges.py::extract_all_grids), shape-prior inference, distances."""

from __future__ import annotations

import re
from typing import Any, Optional

from harness.types import Grid

MAX_GAP = 2  # Land: allow a small gap of noise lines within a grid block
MAX_WIDTH_DIFF = 5  # Land: tolerate ragged rows up to 5 columns apart


def normalize_grid(obj: Any) -> Optional[Grid]:
    """Coerce nested sequences / numpy arrays to a rectangular list[list[int]] or None."""
    if obj is None:
        return None
    if hasattr(obj, "tolist"):
        try:
            obj = obj.tolist()
        except Exception:
            return None
    if not isinstance(obj, list) or len(obj) == 0:
        return None
    rows: Grid = []
    width: Optional[int] = None
    for row in obj:
        if hasattr(row, "tolist"):
            try:
                row = row.tolist()
            except Exception:
                return None
        if not isinstance(row, list) or len(row) == 0:
            return None
        if width is None:
            width = len(row)
        elif len(row) != width:
            return None
        try:
            rows.append([int(c) for c in row])
        except (TypeError, ValueError):
            return None
    return rows


def shape_of(grid: Optional[Grid]) -> Optional[tuple[int, int]]:
    if not grid or not grid[0]:
        return None
    return (len(grid), len(grid[0]))


def grid_key(grid: Grid) -> tuple[tuple[int, ...], ...]:
    return tuple(tuple(int(c) for c in row) for row in grid)


def key_to_grid(key: tuple[tuple[int, ...], ...]) -> Grid:
    return [list(row) for row in key]


def grid_to_csv(grid: Optional[Grid]) -> str:
    """Land's winning format: CSV rows (best of 9 tested; a bad format costs ~10%)."""
    if not grid:
        return ""
    return "\n".join(",".join(str(c) for c in row) for row in grid)


def grid_to_diagram(grid: Grid) -> str:
    """Poetiq <Diagram> style: space-separated cells."""
    return "\n".join(" ".join(str(c) for c in row) for row in grid)


def grids_equal(a: Optional[Grid], b: Optional[Grid]) -> bool:
    if a is None or b is None:
        return a is None and b is None
    return grid_key(a) == grid_key(b)


def normalized_distance(a: Optional[Grid], b: Optional[Grid]) -> float:
    """0 = identical, 1 = maximally different (shape mismatch counts as 1.0)."""
    if a is None or b is None:
        return 1.0
    if shape_of(a) != shape_of(b):
        return 1.0
    total = len(a) * len(a[0])
    if total == 0:
        return 0.0
    diff = sum(1 for ra, rb in zip(a, b) for ca, cb in zip(ra, rb) if ca != cb)
    return diff / total


def _parse_row(stripped: str) -> Optional[list[int]]:
    """Parse one candidate CSV row (Land's tolerant row parser)."""
    if not stripped:
        return None
    if re.match(r"^Row\s+\d+:?$", stripped, re.IGNORECASE):
        return None
    if stripped.startswith(("-", "*", "+")):
        return None
    try:
        clean = stripped.replace("`", " ").replace("[", " ").replace("]", " ").strip()
        numbered = re.match(r"^\d+[.)]\s+", clean)
        if numbered:
            clean = clean[numbered.end():]
        tokens = clean.split(",")
        if tokens and all(t.strip().isdigit() for t in tokens):
            return [int(t.strip()) for t in tokens]
        # Fallback: "Row 1: 8,8,8" / "Output: 1,2,3" — slice first-digit..last-digit
        if ":" in clean:
            clean = clean.split(":")[-1].strip()
        first = re.search(r"\d", clean)
        if first:
            last_digit = max((i for i, ch in enumerate(clean) if ch.isdigit()), default=-1)
            if last_digit >= first.start():
                sub = clean[first.start(): last_digit + 1]
                sub_tokens = sub.split(",")
                if len(sub_tokens) > 1 and all(t.strip().isdigit() for t in sub_tokens):
                    remainder = clean[last_digit + 1:].strip()
                    if not any(c.isalpha() for c in remainder):
                        return [int(t.strip()) for t in sub_tokens]
    except ValueError:
        pass
    return None


def _scan_blocks(text: str) -> list[list[list[int]]]:
    """Group candidate rows into blocks; code fences are hard separators."""
    lines = text.splitlines()
    candidate_rows: list[Optional[list[int]]] = []
    hard_separators: list[int] = []
    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped.startswith("```"):
            candidate_rows.append(None)
            hard_separators.append(i)
            continue
        candidate_rows.append(_parse_row(stripped))

    blocks: list[list[list[int]]] = []
    current: list[list[int]] = []
    last_row = -1
    for i, row in enumerate(candidate_rows):
        if row is None:
            continue
        if not current:
            current = [row]
            last_row = i
            continue
        has_hard_sep = any(last_row < sep < i for sep in hard_separators)
        gap = i - last_row - 1
        width_ok = abs(len(row) - len(current[0])) <= MAX_WIDTH_DIFF
        if has_hard_sep or gap > MAX_GAP or not width_ok:
            blocks.append(current)
            current = [row]
        else:
            current.append(row)
        last_row = i
    if current:
        blocks.append(current)
    return blocks


def extract_all_grids(text: str) -> list[Grid]:
    """All CSV-like grid blocks in a text, in order (Land's judges.extract_all_grids)."""
    if not text:
        return []
    return _scan_blocks(text.strip())


def parse_grid_from_text(text: str) -> Optional[Grid]:
    """The LAST valid grid block in a text (Land: the final answer). None if absent."""
    blocks = extract_all_grids(text)
    return blocks[-1] if blocks else None


def distinct_grids(grids: list[Grid]) -> list[Grid]:
    seen: set[tuple] = set()
    out: list[Grid] = []
    for g in grids:
        k = grid_key(g)
        if k not in seen:
            seen.add(k)
            out.append(g)
    return out


def modal_color(train_out: list[Grid]) -> int:
    """Most frequent cell value across train outputs (0 excluded when possible)."""
    counts: dict[int, int] = {}
    for g in train_out:
        for row in g or []:
            for c in row:
                counts[c] = counts.get(c, 0) + 1
    if not counts:
        return 0
    nonzero = {c: n for c, n in counts.items() if c != 0}
    pool = nonzero or counts
    return max(sorted(pool), key=lambda c: pool[c])


# ---------------------------------------------------------------------------
# Shape prior: the true output shape is a near-deterministic function of the
# train pairs (measured on the eval split: relation consistent in 98.3% of
# tasks; identity-train -> identity-test 117/117). Used by the pass@2
# shape-repair hedge.
# ---------------------------------------------------------------------------

def _axis_candidates(hs_in: list[int], hs_out: list[int]) -> list[tuple[int, int]]:
    """All (a, b) with h_out = a*h_in + b fitting every train pair, a in 0..4."""
    fits: list[tuple[int, int]] = []
    for a in range(0, 5):
        bs = {ho - a * hi for hi, ho in zip(hs_in, hs_out)}
        if len(bs) == 1:
            b = bs.pop()
            if -12 <= b <= 12:
                fits.append((a, b))
    return fits


def suggest_output_shape(
    train_in: list[Grid], train_out: list[Grid], test_input: Grid
) -> Optional[tuple[int, int]]:
    """Predict the test output shape from the train pairs, or None if ambiguous.

    Rule precedence matches the measured eval-split statistics:
      1. identity (all train pairs shape-preserving) -> test input shape
         (measured: 117/117 test outputs identity when train is all-identity);
      2. constant output dims across train -> those dims;
      3. per-axis linear fit h' = a*h + b, a in 0..4, b in [-12, 12], required to
         fit every train pair AND agree on a single test prediction (else None).
    """
    if not train_in or not train_out or len(train_in) != len(train_out):
        return None
    ti_shape = shape_of(test_input)
    if ti_shape is None:
        return None
    in_shapes = [shape_of(g) for g in train_in]
    out_shapes = [shape_of(g) for g in train_out]
    if any(s is None for s in in_shapes) or any(s is None for s in out_shapes):
        return None

    if all(i == o for i, o in zip(in_shapes, out_shapes)):
        return ti_shape
    # Constant-output rule needs >= 2 train pairs: with a single pair it is
    # indistinguishable from input-dependent (e.g. scaling) output dims.
    if len(set(out_shapes)) == 1 and len(train_in) >= 2:
        return out_shapes[0]

    pred: list[int] = []
    for axis in (0, 1):
        ins = [s[axis] for s in in_shapes]
        outs = [s[axis] for s in out_shapes]
        fits = _axis_candidates(ins, outs)
        if not fits:
            return None
        preds = {a * ti_shape[axis] + b for a, b in fits}
        if len(preds) != 1:
            return None  # fits disagree on the test input -> ambiguous
        p = preds.pop()
        if not (1 <= p <= 64):
            return None
        pred.append(p)
    return (pred[0], pred[1])


In [ ]:
%%writefile /kaggle/working/harness/parsing.py
"""Tolerant extraction of code blocks (and transform entry points) from model output.

Local 27B models produce malformed output far more often than frontier APIs:
unfenced code, ```py fences, multiple blocks, prose after code, `def solver`
instead of `def transform`, and refusals. Everything here is best-effort and
returns None instead of raising.
"""

from __future__ import annotations

import re
from typing import Optional

FENCED_RE = re.compile(r"```([a-zA-Z0-9_+-]*)[ \t]*\r?\n(.*?)```", re.DOTALL)
REFUSAL_PATTERNS = (
    r"\bi (?:cannot|can't|am unable to|'m unable to)\b",
    r"\bi'?m sorry\b",
    r"\bas an ai\b",
    r"\bi (?:do not|don't) have (?:the )?(?:ability|access)\b",
    r"\bcannot (?:assist|help|comply)\b",
)


def extract_code_blocks(text: str) -> list[tuple[str, str]]:
    """All fenced blocks as (lang, body), in order. Tolerates ``` without lang."""
    if not text:
        return []
    return [(m.group(1).strip().lower(), m.group(2)) for m in FENCED_RE.finditer(text)]


def _has_entry(body: str) -> bool:
    return re.search(r"^\s*def\s+(transform|solver)\s*\(", body, re.MULTILINE) is not None


def _extract_unfenced(text: str) -> Optional[str]:
    """Last resort: code without any fence. Capture from the first import/def line."""
    m = re.search(r"^(?:import\s+\w+|from\s+\w+|def\s+(?:transform|solver)\s*\()", text, re.MULTILINE)
    if not m:
        return None
    body = text[m.start():]
    lines: list[str] = []
    for line in body.splitlines():
        s = line.strip()
        if lines and s and not line[0].isspace() and not s.startswith(("def ", "import ", "from ", "@", "#")):
            # Non-indented prose after code started -> stop.
            if not s.endswith((":","\\", ")", "]")) and "return" not in s:
                break
        lines.append(line)
    code = "\n".join(lines).strip()
    return code if _has_entry(code) else None


def pick_transform_code(text: str) -> Optional[tuple[str, str]]:
    """Find the best (code, entry_name) pair in a model response.

    Preference order:
      1. python/py fenced block defining transform (last one wins — models often
         draft, then emit a final version);
      2. any fenced block defining transform or solver;
      3. unfenced `def transform` / `def solver` region.
    `solver` is accepted and bound to `transform` by the sandbox shim.
    """
    if not text:
        return None
    blocks = extract_code_blocks(text)
    py_blocks = [b for lang, b in blocks if lang in ("python", "py", "python3")]

    for pool in (py_blocks, [b for _, b in blocks]):
        with_transform = [b for b in pool if re.search(r"^\s*def\s+transform\s*\(", b, re.MULTILINE)]
        if with_transform:
            return with_transform[-1].strip(), "transform"
        with_solver = [b for b in pool if re.search(r"^\s*def\s+solver\s*\(", b, re.MULTILINE)]
        if with_solver:
            return with_solver[-1].strip(), "solver"

    unfenced = _extract_unfenced(text)
    if unfenced:
        entry = "transform" if re.search(r"^\s*def\s+transform\s*\(", unfenced, re.MULTILINE) else "solver"
        return unfenced, entry
    return None


def looks_like_refusal(text: str) -> bool:
    if not text or len(text.strip()) < 3:
        return True  # empty/near-empty treated as refusal
    lowered = text.lower()
    return any(re.search(p, lowered) for p in REFUSAL_PATTERNS)


def truncate(text: str, max_chars: int, marker: str = "\n...[truncated]\n") -> str:
    if not text or len(text) <= max_chars:
        return text or ""
    head = max_chars * 2 // 3
    tail = max_chars - head
    return text[:head] + marker + text[-tail:]


In [ ]:
%%writefile /kaggle/working/harness/budget.py
"""Token + wall-clock budget accounting, and the deadline-driven degradation ladder."""

from __future__ import annotations

import contextvars
import threading
import time
from dataclasses import dataclass
from typing import Optional

# Degradation ladder thresholds (fraction of total wall clock elapsed).
LEVEL_THRESHOLDS = (0.60, 0.75, 0.88, 0.93)

# Current task scope for token accounting; each asyncio.Task gets its own copy
# of the context, so concurrent tasks never cross-attribute tokens.
_CURRENT_TASK: contextvars.ContextVar[Optional[str]] = contextvars.ContextVar(
    "ledger_task", default=None
)


class Deadline:
    """Global wall-clock deadline with a 4-level degradation ladder.

    level(): 0 normal, 1 @60%, 2 @75%, 3 @88%, 4 @93% (stop solving, finalize).
    """

    def __init__(self, total_seconds: float, start: Optional[float] = None):
        self.total_seconds = float(total_seconds)
        self.start = time.monotonic() if start is None else start

    def elapsed(self) -> float:
        return time.monotonic() - self.start

    def remaining(self) -> float:
        return max(0.0, self.total_seconds - self.elapsed())

    def fraction_used(self) -> float:
        if self.total_seconds <= 0:
            return 1.0
        return min(1.0, self.elapsed() / self.total_seconds)

    def level(self) -> int:
        f = self.fraction_used()
        level = 0
        for i, t in enumerate(LEVEL_THRESHOLDS, start=1):
            if f >= t:
                level = i
        return level

    def expired(self) -> bool:
        return self.remaining() <= 0.0

    def task_deadline(self, tasks_remaining: int, cap_s: float = 900.0) -> float:
        """Per-task soft deadline: fair share of remaining time x1.5 front-load, capped."""
        if tasks_remaining <= 0:
            return 0.0
        return min(cap_s, self.remaining() / tasks_remaining * 1.5)


@dataclass
class ComponentUsage:
    billed_prompt: int = 0
    billed_completion: int = 0
    effective_prompt: int = 0  # prefix-cache-aware: shared prefills counted once
    calls: int = 0

    @property
    def billed(self) -> int:
        return self.billed_prompt + self.billed_completion

    @property
    def effective(self) -> int:
        return self.effective_prompt + self.billed_completion


class TokenLedger:
    """Records per-component token usage; distinguishes billed vs GPU-effective
    tokens (a repeated cache_key prefill within one task is counted once)."""

    def __init__(self, per_task_effective_cap: int = 120_000):
        self.per_task_effective_cap = per_task_effective_cap
        self._lock = threading.Lock()
        self.components: dict[str, ComponentUsage] = {}
        self._seen_by_task: dict[str, set[str]] = {}
        self.task_totals: dict[str, dict[str, int]] = {}

    # -- task scoping ---------------------------------------------------------
    def begin_task(self, task_id: str) -> None:
        _CURRENT_TASK.set(task_id)
        with self._lock:
            self.task_totals.setdefault(task_id, {"billed": 0, "effective": 0})
            self._seen_by_task.setdefault(task_id, set())

    def end_task(self) -> dict[str, int]:
        tid = _CURRENT_TASK.get()
        _CURRENT_TASK.set(None)
        with self._lock:
            totals = dict(self.task_totals.get(tid or "", {"billed": 0, "effective": 0}))
            self._seen_by_task.pop(tid or "", None)
            return totals

    # -- accounting -----------------------------------------------------------
    def add(
        self,
        component: str,
        prompt_tokens: int,
        completion_tokens: int,
        cache_key: Optional[str] = None,
    ) -> None:
        tid = _CURRENT_TASK.get()
        with self._lock:
            c = self.components.setdefault(component, ComponentUsage())
            c.billed_prompt += prompt_tokens
            c.billed_completion += completion_tokens
            c.calls += 1
            effective_prompt = prompt_tokens
            if cache_key is not None:
                seen = self._seen_by_task.setdefault(tid or "", set())
                if cache_key in seen:
                    effective_prompt = 0  # prefix-cache hit: no fresh prefill
                else:
                    seen.add(cache_key)
            c.effective_prompt += effective_prompt
            if tid is not None:
                t = self.task_totals.setdefault(tid, {"billed": 0, "effective": 0})
                t["billed"] += prompt_tokens + completion_tokens
                t["effective"] += effective_prompt + completion_tokens

    # -- queries --------------------------------------------------------------
    def task_effective(self, task_id: Optional[str] = None) -> int:
        tid = task_id if task_id is not None else _CURRENT_TASK.get()
        with self._lock:
            if tid is None:
                return 0
            return self.task_totals.get(tid, {}).get("effective", 0)

    def task_over_budget(self, task_id: Optional[str] = None) -> bool:
        return self.task_effective(task_id) >= self.per_task_effective_cap

    def snapshot(self) -> dict:
        with self._lock:
            return {
                "components": {
                    k: {
                        "billed": v.billed,
                        "effective": v.effective,
                        "calls": v.calls,
                    }
                    for k, v in self.components.items()
                },
                "total_billed": sum(v.billed for v in self.components.values()),
                "total_effective": sum(v.effective for v in self.components.values()),
                "tasks": len(self.task_totals),
            }


In [ ]:
%%writefile /kaggle/working/harness/sandbox.py
"""Sandboxed execution of model-generated code.

Port of Poetiq's arc_agi/sandbox.py (pure asyncio subprocess, clean env, hard
timeout) with additions for local-model robustness:
  - output cap (a runaway print loop cannot OOM the harness),
  - `solver` entry auto-bound to `transform`,
  - optional scipy (Kaggle has it; local dev boxes may not),
  - minimal clean env (no API keys, no HOME) + the few vars Windows needs.
"""

from __future__ import annotations

import asyncio
import json
import os
import sys
import tempfile

OUTPUT_CAP = 64 * 1024  # bytes
DEFAULT_TIMEOUT_S = 5.0  # Poetiq config: timeout_s = 5


def _clean_env() -> dict[str, str]:
    env = {"PYTHONHASHSEED": "0", "PYTHONDONTWRITEBYTECODE": "1"}
    if sys.platform == "win32":
        # numpy/stdlib need these on Windows; still no secrets, no HOME.
        for k in ("SYSTEMROOT", "WINDIR", "TEMP", "TMP", "PATH"):
            v = os.environ.get(k)
            if v:
                env[k] = v
    return env


def _build_script(code: str, entry: str = "transform") -> str:
    # NOTE: template lines are at column 0 on purpose — model code is interpolated
    # verbatim, so any shared-indent dedent would corrupt it.
    return f"""
# generated file
{code}
if __name__ == '__main__':
    import json
    import sys
    import numpy as np
    try:
        import scipy  # optional; present on Kaggle
    except ImportError:
        pass
    _fn = globals().get('transform') or globals().get('solver')
    if _fn is None:
        raise NameError("no transform()/solver() function defined")
    data = json.load(sys.stdin)
    res = _fn(np.array(data['input']))
    import numpy as _np
    arr = _np.asarray(res)
    if arr.ndim != 2:
        raise ValueError("transform must return a 2D grid, got ndim=%d" % arr.ndim)
    print(json.dumps({{"ok": True, "result": arr.astype(int).tolist()}}))
"""


async def run_code(
    code: str,
    input_grid: list,
    timeout_s: float = DEFAULT_TIMEOUT_S,
    output_cap: int = OUTPUT_CAP,
    entry: str = "transform",
) -> tuple[bool, str]:
    """Run `code` in a subprocess on `input_grid`. Returns (ok, json_result_or_error)."""
    script = _build_script(code, entry)
    with tempfile.TemporaryDirectory() as td:
        path = os.path.join(td, "u.py")
        with open(path, "w", encoding="utf-8") as f:
            f.write(script)
        proc = await asyncio.create_subprocess_exec(
            sys.executable,
            path,
            stdin=asyncio.subprocess.PIPE,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE,
            cwd=td,
            env=_clean_env(),
        )
        try:
            stdout_b, stderr_b = await asyncio.wait_for(
                proc.communicate(input=json.dumps({"input": input_grid}).encode()),
                timeout=timeout_s,
            )
        except asyncio.TimeoutError:
            try:
                proc.kill()
            except ProcessLookupError:
                pass
            try:
                await proc.wait()
            except Exception:
                pass
            return False, "timeout"

        stdout = (stdout_b or b"")[:output_cap].decode("utf-8", errors="replace")
        stderr = (stderr_b or b"")[:output_cap].decode("utf-8", errors="replace")

        if proc.returncode != 0:
            return False, (stderr.strip() or stdout.strip() or "execution failed")
        try:
            payload = json.loads(stdout)
            return bool(payload.get("ok")), json.dumps(payload.get("result"))
        except Exception as e:
            return False, f"bad-json: {e}"


def run_code_sync(
    code: str,
    input_grid: list,
    timeout_s: float = DEFAULT_TIMEOUT_S,
    output_cap: int = OUTPUT_CAP,
    entry: str = "transform",
) -> tuple[bool, str]:
    """Synchronous convenience wrapper (used by tests and single-threaded paths)."""
    return asyncio.run(run_code(code, input_grid, timeout_s, output_cap, entry))


In [ ]:
%%writefile /kaggle/working/harness/oracle.py
"""Train-pair oracle: soft scoring, diff diagnostics, feedback construction.

Port of Poetiq's _soft_score / _array_diff / _build_feedback / _eval_on_train_and_test
with local-model hardening (grid normalisation, per-run error capture).

Verification semantics (Land, measured): precision ~97%, recall ~43% — passing all
train pairs is a strong positive signal, failing is a weak negative one, so failed
candidates are never discarded, only down-ranked.
"""

from __future__ import annotations

import asyncio
import json
from typing import Optional

from harness.grids import normalize_grid
from harness.sandbox import DEFAULT_TIMEOUT_S, run_code
from harness.types import Grid, RunResult


def soft_score(pred: Optional[Grid], truth: Grid) -> float:
    """Mean per-cell accuracy; 0.0 on shape mismatch, 1.0 on exact match."""
    if pred is None:
        return 0.0
    if len(pred) != len(truth) or any(len(p) != len(t) for p, t in zip(pred, truth)):
        return 0.0
    total = sum(len(t) for t in truth)
    if total == 0:
        return 1.0
    match = sum(1 for p, t in zip(pred, truth) for cp, ct in zip(p, t) if cp == ct)
    return match / total


def array_diff(pred: Grid, truth: Grid) -> str:
    """Poetiq diff visualisation: correct cells as-is, wrong cells 'pred/correct'."""
    out = []
    for pr, tr in zip(pred, truth):
        row = []
        for cp, ct in zip(pr, tr):
            row.append(str(int(cp)) if cp == ct else f"{int(cp)}/{int(ct)}")
        out.append(" ".join(row))
    return "\n".join(out)


async def evaluate_code(
    code: str,
    train_in: list[Grid],
    train_out: list[Grid],
    test_in: list[Grid],
    timeout_s: float = DEFAULT_TIMEOUT_S,
    entry: str = "transform",
) -> tuple[list[RunResult], list[RunResult]]:
    """Execute `code` on every train pair (scored) and test input (unscored)."""
    train_results: list[RunResult] = []
    for iin, oout in zip(train_in, train_out):
        ok, out_str = await run_code(code, iin, timeout_s=timeout_s, entry=entry)
        rr = RunResult(raw_output=out_str)
        if not ok:
            rr.error = out_str or "Execution failed."
        else:
            try:
                parsed = normalize_grid(json.loads(out_str))
            except Exception:
                parsed = None
            if parsed is None:
                rr.error = f"Failed to parse output as a rectangular 2D int grid. Output was:\n{out_str[:500]}"
            else:
                rr.grid = parsed
                rr.soft_score = soft_score(parsed, oout)
                rr.success = parsed == oout
        train_results.append(rr)

    test_results: list[RunResult] = []
    for iin in test_in:
        ok, out_str = await run_code(code, iin, timeout_s=timeout_s, entry=entry)
        rr = RunResult(raw_output=out_str)
        if not ok:
            rr.error = out_str or "Execution failed."
        else:
            try:
                parsed = normalize_grid(json.loads(out_str))
            except Exception:
                parsed = None
            if parsed is None:
                rr.error = "Failed to parse output as a rectangular 2D int grid."
            else:
                rr.grid = parsed
        test_results.append(rr)
    return train_results, test_results


def evaluate_code_sync(*args, **kwargs) -> tuple[list[RunResult], list[RunResult]]:
    return asyncio.run(evaluate_code(*args, **kwargs))


def build_feedback(
    train_results: list[RunResult], train_out: list[Grid]
) -> tuple[str, float]:
    """Poetiq feedback: per-example diagnosis (shape mismatch / diff grid / error)
    plus a mean soft score in [0, 1]."""
    parts: list[str] = []
    scores: list[float] = []

    for i, rr in enumerate(train_results):
        if rr.success:
            parts.append(f"Solves Example #{i + 1} correctly. ")
            scores.append(1.0)
            continue
        lines = [f"Solves Example #{i + 1} incorrectly. "]
        if rr.grid is None:
            scores.append(0.0)
            lines.append("\nThe output has to be a rectangular grid of numbers.\n")
        else:
            truth = train_out[i]
            if len(rr.grid) != len(truth) or any(
                len(p) != len(t) for p, t in zip(rr.grid, truth)
            ):
                scores.append(0.0)
                pred_shape = (len(rr.grid), len(rr.grid[0]) if rr.grid else 0)
                true_shape = (len(truth), len(truth[0]) if truth else 0)
                lines.append(
                    f"\n\nShape mismatch: your prediction's shape was {pred_shape}, "
                    f"while the correct shape was {true_shape}."
                )
            else:
                lines.append(
                    "\nYour code's output does not match the expected output."
                    "\n\nBelow is a visualization of the 2D array your code produced as well as the expected output.\n"
                    "Correctly predicted values are shown as-is while the incorrectly predicted values are shown "
                    "in the format 'prediction/correct':\n"
                )
                lines.append(f"\n```\n{array_diff(rr.grid, truth)}\n```\n")
                s = soft_score(rr.grid, truth)
                scores.append(s)
                lines.append(f"Output accuracy: {s:.2f} (0 is worst, 1 is best).\n")
        if rr.error:
            lines.append(f"\n\nYour code produced the following error:\n{rr.error[:800]}\n")
        parts.append("".join(lines))

    mean_score = sum(scores) / len(scores) if scores else 0.0
    return "\n\n".join(parts), mean_score


In [ ]:
%%writefile /kaggle/working/harness/pool.py
"""Solution pool: the diversity-preserving memory of a refinement expert.

Poetiq semantics: every scored partial solution is kept; the feedback context
shows the top-K by soft score (K = max_solutions = 5), worst-first when
improving_order is set, so the next sample is conditioned on a *distribution*
of hypotheses rather than a single chain (see HYPOTHESIS.md §2).
"""

from __future__ import annotations

import hashlib
import re

from harness.prompts import build_feedback_block


def _code_fingerprint(code: str) -> str:
    norm = re.sub(r"\s+", " ", code.strip())
    return hashlib.sha1(norm.encode("utf-8")).hexdigest()


class SolutionPool:
    def __init__(self, max_solutions: int = 5, improving_order: bool = True):
        self.max_solutions = max_solutions
        self.improving_order = improving_order
        self._entries: list[dict] = []
        self._fingerprints: set[str] = set()

    def __len__(self) -> int:
        return len(self._entries)

    def add(self, code: str, feedback: str, score: float) -> bool:
        """Add a scored partial solution. Returns False if an identical program
        (whitespace-normalised) is already pooled — duplicate blocks would waste
        feedback context without adding diversity."""
        fp = _code_fingerprint(code)
        if fp in self._fingerprints:
            return False
        self._fingerprints.add(fp)
        self._entries.append({"code": code, "feedback": feedback, "score": float(score)})
        return True

    @property
    def entries(self) -> list[dict]:
        return list(self._entries)

    def best(self) -> dict | None:
        if not self._entries:
            return None
        return max(self._entries, key=lambda e: e["score"])

    def feedback_block(self) -> str:
        """Top-K scored entries rendered as the Poetiq <solution_i> block."""
        return build_feedback_block(
            self._entries, max_examples=self.max_solutions, improving_order=self.improving_order
        )


In [ ]:
%%writefile /kaggle/working/harness/prompts.py
"""Prompt construction.

- Expert A: Poetiq SOLVER_PROMPT_2, reused nearly verbatim (verified 54% engine;
  prescriptive template is safe *because* it runs inside the pooled/shuffled
  refinement loop — see HYPOTHESIS.md §2).
- Expert B: Land-style minimal codegen prompt (v1b lineage) with test inputs as
  generalization probes — deliberately NOT prescriptive (Land's negative result).
- Feedback: Poetiq FEEDBACK_PROMPT verbatim + scored pool block (improving order).
- Text channel: Land's minimal free-form prompt (build_prompt lineage) + CSV answer.
- Judge: port of Land's build_duo_pick_prompt (holistic, all traces, top-2 grids,
  synthesis allowed, anti-consistency closing instructions preserved).
"""

from __future__ import annotations

from typing import Optional

from harness.grids import grid_to_csv, grid_to_diagram
from harness.types import Grid

# ---------------------------------------------------------------------------
# Expert A — Poetiq SOLVER_PROMPT_2 (verbatim from reference/poetiq-solver)
# ---------------------------------------------------------------------------

SOLVER_PROMPT_A = '''
You are a world-class expert in solving Abstract Reasoning Corpus (ARC) tasks. Your approach is methodical, creative, and highly effective. You are also a master Python coder, producing elegant, efficient, and well-documented solutions.

Your goal is to analyze a set of input-output examples and devise a Python function that accurately transforms any input grid into its corresponding output grid. The key is to identify a *single, consistent transformation rule* that generalizes across *all* examples. Do not give up until you find a correct solution.

Follow this iterative process:

**Part 1: Initial Analysis and Hypothesis Generation**

1.  **Example Inspection:** Carefully examine the input and output grids for each example. Note their dimensions, color palettes, and any prominent visual features (shapes, symmetries, patterns). Use visualization techniques to aid your analysis.
2.  **Transformation Hypotheses:** Formulate several candidate transformation rules. Start with simpler rules and gradually increase complexity. Consider these categories:
    *   **Color Transformations:** Replacing colors based on specific criteria (e.g., adjacency, frequency). For example, replace all 0s with 1s, or replace the most frequent color with the least frequent color.
    *   **Object Isolation:** Identifying and isolating objects based on color, shape, or position. For example, extract the largest connected component of a certain color, or isolate objects based on their spatial relationships.
    *   **Spatial Operations:** Rotating, reflecting, resizing, or moving objects. For example, rotate the grid by 90 degrees, reflect the grid horizontally or vertically, or resize the grid by a certain factor.
    *   **Pattern Generation:** Replicating or extending existing patterns. For example, repeat a certain pattern across the grid, or generate a new pattern based on the existing patterns.
3.  **Symmetry Analysis:** Identify any symmetries (rotational, reflectional) in the input and output grids. Determine if the transformation preserves or alters these symmetries.

**Part 2: Iterative Testing and Refinement**

1.  **Code Implementation:** Implement your strongest candidate rule as a Python function. The function *must* accept a 2D numpy array as input and return a 2D numpy array as output.
2.  **Rigorous Testing:** Test your code against *all* training examples. A single failure indicates an incorrect rule.
3.  **Feedback Analysis:** If your code fails, carefully analyze the feedback. Identify the specific examples that failed and the nature of the errors. Use print statements to debug intermediate values and verify your assumptions.
4.  **Hypothesis Refinement:** Based on the feedback, refine your transformation rule. This may involve adjusting parameters, adding new conditions, or discarding the rule altogether and starting with a new hypothesis.
5.  **Repeat:** Continue this iterative process of coding, testing, and refining until you find a rule that works for all training examples. Do not give up until you find a correct solution.

**Part 3: Coding Guidelines**

1.  **Available Libraries:** You can use `numpy`, `cv2` (OpenCV), and any library from the standard Python library.
2.  **Computer Vision Techniques:** Consider using `cv2` for tasks involving object detection, edge detection, or image filtering.
3.  **Utility Functions:** Write reusable utility functions to improve code modularity and readability.
4.  **Error Handling:** Implement robust error handling to gracefully manage edge cases and invalid inputs.
5.  **Code Clarity:** Write clean, well-documented code with meaningful variable names and comments.

**Part 4: Output Requirements**

1.  **Output Format:**
    *   Begin with a concise paragraph explaining the proposed solution, followed by a Python code section.
    *   You *must* provide a code output representing your best attempt. Do not give up or refuse to produce code.
    *   **The code section must be a single, valid Python code block in markdown fenced code block format and nothing else.**
    *   The main transform function must have the signature `def transform(grid: np.ndarray) -> np.ndarray`.
    *   Document the transformation rule implemented in the docstring of the transform function.
    *   Do not include any `__name__ == "__main__"` block. This will be added later by the user. You are writing a library function.

**Example:**

**Problem:**
Input:
<Diagram>
0 0 1
0 1 0
1 0 0
</Diagram>

Output:
<Diagram>
1 1 1
1 1 1
1 1 1
</Diagram>

**Explanation:**
Replace all 0s with 1s.

```python
import numpy as np

def transform(grid: np.ndarray) -> np.ndarray:
    """Replace all 0s with 1s."""
    return np.where(grid == 0, 1, grid)
```

**PROBLEM:**

Below is a textual representation of the input-output examples and the challenge to be solved.

$$problem$$
'''

# ---------------------------------------------------------------------------
# Expert B — Land-style minimal codegen prompt (no hypothesis categories,
# no output template beyond the code contract; probes = test inputs)
# ---------------------------------------------------------------------------

SOLVER_PROMPT_B_HEADER = '''Below is an ARC AGI task. You're given the training input/output pairs. Your task is to write a python function transform(grid) that returns the output grid. The grid is a 2D numpy array. The transform() function must solve all the input/output pairs. You're also given the input-only test data to help you ensure your solution is generalizable.

You may use numpy, scipy, and cv2 (OpenCV) for grid manipulation.
'''

SOLVER_PROMPT_B_FOOTER = '''
Write a brief explanation of the rule you found, then output the python code for the transform() function in a single markdown code block. Do not include any `__name__ == "__main__"` block.
'''

# ---------------------------------------------------------------------------
# Feedback (Poetiq verbatim)
# ---------------------------------------------------------------------------

FEEDBACK_PROMPT = '''
**EXISTING PARTIAL/INCORRECT SOLUTIONS:**

Following are some of the best, though not completely correct, solutions so far. For each solution, its code, corresponding feedback regarding its output on the example problems, and a numeric score between 0. (worst) and 1. (best) indicating the quality of outputs is also provided. Study these solutions and corresponding feedback and produce a new solution fixing all the issues. Make sure to follow the output format specified earlier.

$$feedback$$
'''

FEEDBACK_SOLUTION_TEMPLATE = """
<solution_{index}>
<solution_code>
```python
{code}
```
</solution_code>
<solution_evaluation>
{feedback}
</solution_evaluation>
<solution_score>
{score}
</solution_score>
</solution_{index}>
"""

# ---------------------------------------------------------------------------
# Text channel — Land's minimal free-form prompt (prompts_standard.build_prompt
# lineage) + the CSV answer format (winner of Land's 9-format benchmark).
# ---------------------------------------------------------------------------

TEXT_PROMPT_HEADER = '''You are solving an ARC (Abstraction and Reasoning Corpus) task.
Each grid cell is an integer 0-9 representing a color.
Use the solved examples to infer the transformation and apply it to the test input.
'''

TEXT_PROMPT_FOOTER = '''
Respond with an explanation of your thinking that is detailed enough that someone can reconstruct your solution. Afterwards, you MUST also respond with the completed output grid.
Format every grid you output as comma-separated values (CSV) with each row on a new line, inside its own fenced code block, like this:
```
7,0,0,7
0,7,7,0
```
'''


def _diagram(grid: Grid) -> str:
    return grid_to_diagram(grid)


def build_problem_str(
    train_in: list[Grid],
    train_out: list[Grid],
    test_in: list[Grid],
    shuffle: bool = True,
    seed: int = 0,
) -> str:
    """Poetiq format_problem: <Diagram> examples + challenges, train order shuffled
    with `seed` (Poetiq passes seed+iteration each round: free augmentation)."""
    order = list(range(len(train_in)))
    if shuffle and len(order) > 1:
        import random

        rng = random.Random(seed)
        rng.shuffle(order)

    parts: list[str] = []
    for n, i in enumerate(order, start=1):
        parts.append(
            f"""
Example #{n}
Input:
<Diagram>
{_diagram(train_in[i])}
</Diagram>

Output:
<Diagram>
{_diagram(train_out[i])}
</Diagram>
"""
        )
    for n, tin in enumerate(test_in, start=1):
        parts.append(
            f"""
Challenge #{n}
Input:
<Diagram>
{_diagram(tin)}
</Diagram>
"""
        )
    return "".join(parts)


def build_solver_a_prompt(
    train_in: list[Grid], train_out: list[Grid], test_in: list[Grid],
    shuffle: bool = True, seed: int = 0,
) -> str:
    problem = build_problem_str(train_in, train_out, test_in, shuffle=shuffle, seed=seed)
    return SOLVER_PROMPT_A.replace("$$problem$$", problem)


def build_solver_b_prompt(
    train_in: list[Grid], train_out: list[Grid], test_in: list[Grid],
    shuffle: bool = True, seed: int = 0,
) -> str:
    order = list(range(len(train_in)))
    if shuffle and len(order) > 1:
        import random

        rng = random.Random(seed)
        rng.shuffle(order)
    lines = [SOLVER_PROMPT_B_HEADER, "", "Solved examples:"]
    for n, i in enumerate(order, start=1):
        lines.append(f"Example {n}:")
        lines.append("input:")
        lines.append(grid_to_csv(train_in[i]))
        lines.append("output:")
        lines.append(grid_to_csv(train_out[i]))
        lines.append("")
    lines.append("Input-only test data:")
    for n, tin in enumerate(test_in, start=1):
        lines.append(f"Probe {n}:")
        lines.append("input:")
        lines.append(grid_to_csv(tin))
        lines.append("")
    lines.append(SOLVER_PROMPT_B_FOOTER)
    return "\n".join(lines)


def build_feedback_block(
    solutions: list[dict], max_examples: int = 5, improving_order: bool = True
) -> str:
    """Poetiq create_examples: top-K pool members by score; improving_order shows
    worst->best so the strongest exemplar occupies the recency slot."""
    if not solutions:
        return ""
    ranked = sorted(range(len(solutions)), key=lambda i: solutions[i]["score"], reverse=True)
    ranked = ranked[:max_examples]
    if improving_order:
        ranked = ranked[::-1]
    blocks = []
    for k, idx in enumerate(ranked, start=1):
        s = solutions[idx]
        blocks.append(
            FEEDBACK_SOLUTION_TEMPLATE.format(
                index=k, code=s["code"], feedback=s["feedback"], score=f"{s['score']:.2f}"
            )
        )
    return "\n".join(blocks)


def build_feedback_prompt(solutions: list[dict], max_examples: int = 5, improving_order: bool = True) -> str:
    block = build_feedback_block(solutions, max_examples, improving_order)
    return FEEDBACK_PROMPT.replace("$$feedback$$", block)


def build_text_prompt(
    train_in: list[Grid], train_out: list[Grid], test_input: Grid
) -> str:
    lines = [TEXT_PROMPT_HEADER, "", "Solved examples:"]
    for n, (tin, tout) in enumerate(zip(train_in, train_out), start=1):
        lines.append(f"Example {n}:")
        lines.append("input:")
        lines.append(grid_to_csv(tin))
        lines.append("output:")
        lines.append(grid_to_csv(tout))
        lines.append("")
    lines.append("Test input:")
    lines.append(grid_to_csv(test_input))
    lines.append("")
    lines.append(TEXT_PROMPT_FOOTER)
    return "\n".join(lines)


# ---------------------------------------------------------------------------
# Holistic judge — port of Land's build_duo_pick_prompt (audit_prompts.py)
# ---------------------------------------------------------------------------

JUDGE_CLOSING = """Your task is to understand these solutions, and assess how well they've understood the problem, and how likely their solutions are to provide the correct solution to the test input.
Often, new mechanics are introduced in the test example for which the solutions do not generalize well. Please output two solutions that you think represent the right mechanic for solving the problem.
Output your two solutions as grids (in code blocks). Format the grids as comma-separated values (CSV) with each row on a new line, like this:
```
7,0,0,7
0,7,7,0
```
Explain how you came to these two solutions being the two most likely. In coming up with your two solutions, study all the provided solutions and their reasoning to come up with a meta-conclusion about how to solve the problem.
If you believe none of the provided solutions is right, you may synthesize a new grid recombining the correct sub-components of the flawed solutions.
Put your two final grids at the very END of your answer, one per fenced code block, first choice first, second choice second."""


def build_judge_prompt(
    train_in: list[Grid],
    train_out: list[Grid],
    test_input: Grid,
    entries: list[dict],
    total_attempts: int,
    trace_cap_chars: int = 2500,
) -> str:
    """One long-context prompt containing the task and ALL candidate traces.

    entries: [{"content": trace/code text, "grid": predicted grid}] already
    deduplicated/capped by the caller.
    """
    from harness.parsing import truncate

    parts: list[str] = []
    parts.append(f"Below is a prompt that was run {total_attempts} times:")
    base = build_text_prompt(train_in, train_out, test_input)
    parts.append("\n<PROMPT START>")
    parts.append(base)
    parts.append("<PROMPT STOP>\n")
    parts.append(
        f"Solutions were generated {len(entries)} times, using different types of solvers. "
        "All solutions are represented below:\n"
    )
    for i, e in enumerate(entries, start=1):
        parts.append(f"<SOLUTION {i} START>")
        parts.append("<CONTENT>")
        parts.append(truncate(e.get("content", ""), trace_cap_chars))
        parts.append("</CONTENT>")
        parts.append("<PREDICTED_GRID>")
        parts.append(grid_to_csv(e.get("grid")))
        parts.append("</PREDICTED_GRID>")
        parts.append(f"<SOLUTION {i} STOP>\n")
    parts.append(JUDGE_CLOSING)
    return "\n".join(parts)


In [ ]:
%%writefile /kaggle/working/harness/llm.py
"""LLM clients: an injectable interface with three backends.

- OpenAICompatClient: any OpenAI-compatible endpoint (vLLM). Stdlib-only
  (urllib via asyncio.to_thread) so it works in an offline Kaggle rerun with no
  extra packages. Passes reasoning_effort through chat_template_kwargs
  (vLLM/qwen3 convention; only {xhigh, medium, low} are legal — `high` is HTTP 500).
- LiteLLMClient: keeps Poetiq's acompletion path working (optional import).
- MockLLM: deterministic scriptable backend for tests and dry runs.
"""

from __future__ import annotations

import asyncio
import json
import random
import re
import time
import urllib.error
import urllib.request
from abc import ABC, abstractmethod
from typing import Callable, Optional

from harness.budget import TokenLedger
from harness.types import LLMResponse

VALID_EFFORTS = ("xhigh", "medium", "low")


class LLMClient(ABC):
    @abstractmethod
    async def complete(
        self,
        prompt: str,
        *,
        tag: str = "generic",
        temperature: float = 1.0,
        max_tokens: int = 2048,
        reasoning_effort: Optional[str] = "low",
        timeout: float = 300.0,
        cache_key: Optional[str] = None,
    ) -> LLMResponse:
        """Single-turn completion. `tag` drives accounting/mocking; `cache_key`
        marks requests sharing an identical prefix (prefix-cache accounting)."""


def _check_effort(effort: Optional[str]) -> Optional[str]:
    if effort is None:
        return None
    if effort not in VALID_EFFORTS:
        raise ValueError(
            f"reasoning_effort={effort!r} invalid: Qwen3.8 accepts only {VALID_EFFORTS} "
            "('high' returns HTTP 500 upstream)"
        )
    return effort


class OpenAICompatClient(LLMClient):
    """vLLM / OpenAI-compatible chat completions via stdlib urllib."""

    def __init__(
        self,
        base_url: str = "http://127.0.0.1:8000/v1",
        model: str = "qwen3.8-27b",
        api_key: str = "EMPTY",
        ledger: Optional[TokenLedger] = None,
        retries: int = 2,
        retry_delay: float = 3.0,
    ):
        self.base_url = base_url.rstrip("/")
        self.model = model
        self.api_key = api_key
        self.ledger = ledger
        self.retries = retries
        self.retry_delay = retry_delay

    def _post_sync(self, payload: dict, timeout: float) -> dict:
        req = urllib.request.Request(
            self.base_url + "/chat/completions",
            data=json.dumps(payload).encode("utf-8"),
            headers={
                "Content-Type": "application/json",
                "Authorization": f"Bearer {self.api_key}",
            },
            method="POST",
        )
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            return json.loads(resp.read().decode("utf-8"))

    async def complete(
        self,
        prompt: str,
        *,
        tag: str = "generic",
        temperature: float = 1.0,
        max_tokens: int = 2048,
        reasoning_effort: Optional[str] = "low",
        timeout: float = 300.0,
        cache_key: Optional[str] = None,
    ) -> LLMResponse:
        effort = _check_effort(reasoning_effort)
        payload: dict = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "max_tokens": max_tokens,
            "stream": False,
        }
        if effort is not None:
            payload["chat_template_kwargs"] = {"reasoning_effort": effort}

        last_err: Optional[str] = None
        start = time.monotonic()
        for attempt in range(self.retries + 1):
            try:
                data = await asyncio.to_thread(self._post_sync, payload, timeout)
                text = (data.get("choices") or [{}])[0].get("message", {}).get("content") or ""
                usage = data.get("usage") or {}
                resp = LLMResponse(
                    text=text.strip(),
                    prompt_tokens=int(usage.get("prompt_tokens", 0)),
                    completion_tokens=int(usage.get("completion_tokens", 0)),
                    duration=time.monotonic() - start,
                )
                if self.ledger is not None:
                    self.ledger.add(tag, resp.prompt_tokens, resp.completion_tokens, cache_key=cache_key)
                return resp
            except (urllib.error.URLError, urllib.error.HTTPError, TimeoutError, OSError, json.JSONDecodeError) as e:
                last_err = f"{type(e).__name__}: {e}"
                if attempt < self.retries:
                    await asyncio.sleep(self.retry_delay * (attempt + 1))
        return LLMResponse(error=last_err or "request failed", duration=time.monotonic() - start)


class LiteLLMClient(LLMClient):
    """Poetiq's original path: LiteLLM acompletion (model e.g. 'openai/qwen3.8-27b'
    with api_base pointed at a local vLLM server, or any provider)."""

    def __init__(self, model: str, ledger: Optional[TokenLedger] = None, **props):
        import litellm  # optional dependency; raises ImportError if absent

        self._litellm = litellm
        litellm.suppress_debug_info = True
        self.model = model
        self.ledger = ledger
        self.props = props

    async def complete(
        self,
        prompt: str,
        *,
        tag: str = "generic",
        temperature: float = 1.0,
        max_tokens: int = 2048,
        reasoning_effort: Optional[str] = "low",
        timeout: float = 300.0,
        cache_key: Optional[str] = None,
    ) -> LLMResponse:
        kwargs = dict(self.props)
        if reasoning_effort is not None:
            _check_effort(reasoning_effort)
            kwargs["chat_template_kwargs"] = {"reasoning_effort": reasoning_effort}
        start = time.monotonic()
        try:
            resp = await self._litellm.acompletion(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
                timeout=timeout,
                num_retries=2,
                **kwargs,
            )
            usage = getattr(resp, "usage", None)
            out = LLMResponse(
                text=(resp["choices"][0]["message"]["content"] or "").strip(),
                prompt_tokens=int(getattr(usage, "prompt_tokens", 0) or 0),
                completion_tokens=int(getattr(usage, "completion_tokens", 0) or 0),
                duration=time.monotonic() - start,
            )
            if self.ledger is not None:
                self.ledger.add(tag, out.prompt_tokens, out.completion_tokens, cache_key=cache_key)
            return out
        except Exception as e:  # noqa: BLE001 — client must never kill a task
            return LLMResponse(error=f"{type(e).__name__}: {e}", duration=time.monotonic() - start)


# ---------------------------------------------------------------------------
# MockLLM
# ---------------------------------------------------------------------------

class MockLLM(LLMClient):
    """Deterministic mock for tests/dry runs.

    Two composable modes:
      - scripted: `responses` is a list (consumed cyclically or per-tag dict) of
        str | Exception | callable(prompt, tag) -> str;
      - lookup: built via MockLLM.from_tasks(challenges, solutions) — solver
        prompts are answered with a program that exactly reproduces the known
        outputs (dispatch on exact input match), judge prompts echo the two most
        frequent <PREDICTED_GRID> grids, text prompts echo the test input.
        `fail_every` injects a malformed response every k-th solver call to
        exercise the parsing/refinement paths.
    """

    def __init__(
        self,
        responses: Optional[list | dict] = None,
        lookup: Optional[dict] = None,
        fail_every: int = 0,
        seed: int = 0,
        ledger: Optional[TokenLedger] = None,
        latency: float = 0.0,
    ):
        self.responses = responses
        self.lookup = lookup or {}
        self.fail_every = fail_every
        self.rng = random.Random(seed)
        self.ledger = ledger
        self.latency = latency
        self.calls: list[dict] = []
        self._script_idx = 0
        self._solver_calls = 0

    @classmethod
    def from_tasks(
        cls,
        challenges: dict,
        solutions: Optional[dict] = None,
        fail_every: int = 0,
        seed: int = 0,
        ledger: Optional[TokenLedger] = None,
    ) -> "MockLLM":
        lookup = {}
        for tid, task in challenges.items():
            lookup[tid] = {
                "train_in": [ex["input"] for ex in task["train"]],
                "train_out": [ex["output"] for ex in task["train"]],
                "test_in": [ex["input"] for ex in task["test"]],
                "outputs": (solutions or {}).get(tid),
            }
        return cls(lookup=lookup, fail_every=fail_every, seed=seed, ledger=ledger)

    # -- response construction ---------------------------------------------
    def _solve_code(self, t: dict) -> str:
        pairs = list(zip(t["train_in"], t["train_out"]))
        if t["outputs"]:
            pairs += list(zip(t["test_in"], t["outputs"]))
        table = ",\n    ".join(f"({json.dumps(i)}, {json.dumps(o)})" for i, o in pairs)
        return (
            "import numpy as np\n\n"
            "def transform(grid: np.ndarray) -> np.ndarray:\n"
            '    """Mock lookup solution."""\n'
            "    table = [\n    " + table + ",\n    ]\n"
            "    g = np.asarray(grid).tolist()\n"
            "    for i, o in table:\n"
            "        if g == i:\n"
            "            return np.array(o)\n"
            "    return np.asarray(grid)\n"
        )

    def _identify_task(self, prompt: str) -> Optional[dict]:
        from harness.grids import grid_to_csv, grid_to_diagram

        for tid, t in self.lookup.items():
            for g in t["test_in"]:
                if grid_to_diagram(g) in prompt or grid_to_csv(g) in prompt:
                    return t
        return None

    def _mock_solver_response(self, prompt: str) -> str:
        t = self._identify_task(prompt)
        if t is None or not t["outputs"]:
            return "I could not find a rule.\n```python\nimport numpy as np\ndef transform(grid):\n    return grid\n```"
        code = self._solve_code(t)
        return (
            "The rule maps each known input to its observed output.\n\n"
            f"```python\n{code}\n```"
        )

    def _mock_text_response(self, prompt: str) -> str:
        t = self._identify_task(prompt)
        from harness.grids import grid_to_csv

        if t is None:
            return "0,0\n0,0"
        g = t["test_in"][0]
        return (
            "The pattern looks like it preserves the layout.\n\n"
            "Final answer:\n```\n" + grid_to_csv(g) + "\n```"
        )

    def _mock_judge_response(self, prompt: str) -> str:
        from harness.grids import grid_to_csv

        blocks = re.findall(r"<PREDICTED_GRID>\n(.*?)\n</PREDICTED_GRID>", prompt, re.DOTALL)
        counts: dict[str, int] = {}
        for b in blocks:
            counts[b] = counts.get(b, 0) + 1
        # most frequent first; stable order for ties
        ranked = sorted(counts.items(), key=lambda kv: (-kv[1], blocks.index(kv[0])))
        picks = [g for g, _ in ranked[:2]]
        while len(picks) < 2:
            picks.append("0,0")
        return (
            "Meta-conclusion: the majority mechanic is the most plausible.\n\n"
            "First choice:\n```\n" + picks[0] + "\n```\n\n"
            "Second choice:\n```\n" + picks[1] + "\n```"
        )

    async def complete(
        self,
        prompt: str,
        *,
        tag: str = "generic",
        temperature: float = 1.0,
        max_tokens: int = 2048,
        reasoning_effort: Optional[str] = "low",
        timeout: float = 300.0,
        cache_key: Optional[str] = None,
    ) -> LLMResponse:
        if reasoning_effort is not None:
            _check_effort(reasoning_effort)
        if self.latency:
            await asyncio.sleep(self.latency)
        self.calls.append({"tag": tag, "prompt_chars": len(prompt), "cache_key": cache_key})

        # scripted mode takes priority
        if self.responses is not None:
            item: object
            if isinstance(self.responses, dict):
                pool = self.responses.get(tag) or self.responses.get("*")
                item = pool[self._script_idx % len(pool)] if isinstance(pool, list) else pool
            else:
                item = self.responses[self._script_idx % len(self.responses)]
            self._script_idx += 1
            if isinstance(item, Exception):
                return LLMResponse(error=str(item))
            text = item(prompt, tag) if callable(item) else str(item)
        else:
            self._solver_calls += 1
            if (
                self.fail_every
                and tag.startswith("solver")
                and self._solver_calls % self.fail_every == 0
            ):
                text = "Here is my answer but no code block at all, just prose."
            elif tag.startswith("solver"):
                text = self._mock_solver_response(prompt)
            elif tag == "judge":
                text = self._mock_judge_response(prompt)
            elif tag == "text":
                text = self._mock_text_response(prompt)
            else:
                text = "0,0"

        pt, ct = max(1, len(prompt) // 4), max(1, len(text) // 4)
        if self.ledger is not None:
            self.ledger.add(tag, pt, ct, cache_key=cache_key)
        return LLMResponse(text=text.strip(), prompt_tokens=pt, completion_tokens=ct, duration=0.0)


In [ ]:
%%writefile /kaggle/working/harness/experts.py
"""Candidate-generation channels.

CodeExpert: the Poetiq refinement loop (solve_coding.py port) — shuffled train
order per iteration, pool-5 scored feedback, sandbox after every sample,
early-exit on full train verification, per-iteration budget/deadline checks.

TextChannel: Land's minimal free-form text candidates (no refinement, no
prescriptive template) for raw hypothesis diversity.
"""

from __future__ import annotations

import asyncio
from dataclasses import dataclass
from typing import Optional

from harness.budget import Deadline, TokenLedger
from harness.grids import normalize_grid, parse_grid_from_text
from harness.llm import LLMClient
from harness.oracle import build_feedback, evaluate_code
from harness.parsing import looks_like_refusal, pick_transform_code
from harness.pool import SolutionPool
from harness.prompts import (
    build_feedback_prompt,
    build_solver_a_prompt,
    build_solver_b_prompt,
    build_text_prompt,
)
from harness.types import Candidate, Grid


@dataclass
class ExpertConfig:
    name: str = "expert_a"
    variant: str = "A"  # "A" = Poetiq prescriptive, "B" = Land minimal
    max_iterations: int = 8
    seed: int = 0
    temperature: float = 1.0
    reasoning_effort: str = "low"
    max_tokens: int = 2048
    request_timeout: float = 300.0
    pool_size: int = 5
    shuffle_examples: bool = True
    sandbox_timeout: float = 5.0


@dataclass
class TaskData:
    task_id: str
    train_in: list[Grid]
    train_out: list[Grid]
    test_in: list[Grid]


class CodeExpert:
    def __init__(self, cfg: ExpertConfig):
        self.cfg = cfg

    def _solver_prompt(self, task: TaskData, iteration: int) -> str:
        seed = self.cfg.seed + iteration  # Poetiq: seed + it
        if self.cfg.variant == "A":
            return build_solver_a_prompt(
                task.train_in, task.train_out, task.test_in,
                shuffle=self.cfg.shuffle_examples, seed=seed,
            )
        return build_solver_b_prompt(
            task.train_in, task.train_out, task.test_in,
            shuffle=self.cfg.shuffle_examples, seed=seed,
        )

    async def run(
        self,
        task: TaskData,
        client: LLMClient,
        ledger: Optional[TokenLedger] = None,
        deadline: Optional[Deadline] = None,
        task_deadline_s: Optional[float] = None,
    ) -> list[Candidate]:
        cfg = self.cfg
        pool = SolutionPool(max_solutions=cfg.pool_size)
        candidates: list[Candidate] = []
        start = asyncio.get_event_loop().time()

        for it in range(cfg.max_iterations):
            if deadline is not None and (deadline.level() >= 4 or deadline.expired()):
                break
            if task_deadline_s is not None and asyncio.get_event_loop().time() - start > task_deadline_s:
                break
            if ledger is not None and ledger.task_over_budget(task.task_id):
                break

            prompt = self._solver_prompt(task, it)
            if len(pool):
                prompt += "\n\n" + build_feedback_prompt(pool.entries, max_examples=cfg.pool_size)

            resp = await client.complete(
                prompt,
                tag=f"solver_{cfg.name}",
                temperature=cfg.temperature,
                max_tokens=cfg.max_tokens,
                reasoning_effort=cfg.reasoning_effort,
                timeout=cfg.request_timeout,
                # no cache_key: the prompt changes every iteration (shuffle +
                # pool feedback), so prefill is genuinely re-paid; only the
                # judge council shares an identical prompt.
            )
            if not resp.ok or looks_like_refusal(resp.text):
                continue

            picked = pick_transform_code(resp.text)
            if picked is None:
                continue  # malformed output: skip iteration (Poetiq semantics)
            code, entry = picked

            train_res, test_res = await evaluate_code(
                code, task.train_in, task.train_out, task.test_in,
                timeout_s=cfg.sandbox_timeout, entry=entry,
            )
            verified = bool(train_res) and all(r.success for r in train_res)
            mean_soft = (
                sum(r.soft_score for r in train_res) / len(train_res) if train_res else 0.0
            )
            feedback = ""
            if not verified:
                feedback, _score = build_feedback(train_res, task.train_out)

            candidates.append(
                Candidate(
                    source=cfg.name,
                    iteration=it + 1,
                    trace=resp.text,
                    code=code,
                    grids=[r.grid for r in test_res],
                    verified=verified,
                    soft=mean_soft,
                    feedback=feedback,
                )
            )
            if verified:
                # Poetiq early-exit: a fully verified program is ~97% precise
                # (Land's measurement) — stop spending tokens on this expert.
                return candidates

            if feedback:
                pool.add(code, feedback, mean_soft)

        return candidates


class TextChannel:
    """k free-form text samples per test input, minimal prompt (Land)."""

    def __init__(
        self,
        k: int = 3,
        temperature: float = 1.0,
        reasoning_effort: str = "low",
        max_tokens: int = 2500,
        request_timeout: float = 300.0,
    ):
        self.k = k
        self.temperature = temperature
        self.reasoning_effort = reasoning_effort
        self.max_tokens = max_tokens
        self.request_timeout = request_timeout

    async def run(
        self,
        task: TaskData,
        client: LLMClient,
        ledger: Optional[TokenLedger] = None,
        deadline: Optional[Deadline] = None,
    ) -> list[Candidate]:
        candidates: list[Candidate] = []

        async def one(test_idx: int, sample: int) -> Optional[Candidate]:
            prompt = build_text_prompt(task.train_in, task.train_out, task.test_in[test_idx])
            resp = await client.complete(
                prompt,
                tag="text",
                temperature=self.temperature,
                max_tokens=self.max_tokens,
                reasoning_effort=self.reasoning_effort,
                timeout=self.request_timeout,
            )
            if not resp.ok or looks_like_refusal(resp.text):
                return None
            grid = parse_grid_from_text(resp.text)
            grid = normalize_grid(grid)
            if grid is None:
                return None
            grids: list[Optional[Grid]] = [None] * len(task.test_in)
            grids[test_idx] = grid
            return Candidate(
                source=f"text[{test_idx}]",
                iteration=sample + 1,
                trace=resp.text,
                grids=grids,
                verified=False,
                soft=0.0,
            )

        jobs = [
            one(ti, s)
            for ti in range(len(task.test_in))
            for s in range(self.k)
        ]
        for fut in asyncio.as_completed([asyncio.ensure_future(j) for j in jobs]):
            try:
                cand = await fut
            except Exception:  # noqa: BLE001 — a dead channel never kills the task
                cand = None
            if cand is not None:
                candidates.append(cand)
        return candidates


In [ ]:
%%writefile /kaggle/working/harness/judge.py
"""Land's holistic trace judge: ONE long-context prompt with ALL candidate
traces per test input, a council of N samples of the same model at higher
reasoning effort, each picking its top-2 grids (synthesis allowed), aggregated
2 points per first choice / 1 point per second choice per distinct grid.
"""

from __future__ import annotations

import asyncio
from typing import Optional

from harness.budget import Deadline, TokenLedger
from harness.grids import distinct_grids, extract_all_grids, grid_key, normalize_grid
from harness.llm import LLMClient
from harness.prompts import build_judge_prompt
from harness.types import Candidate, Grid, JudgePick, ScoreEntry


def build_judge_entries(
    candidates: list[Candidate],
    test_idx: int,
    max_entries: int = 12,
) -> list[dict]:
    """Distinct grids for one test input with a representative trace each.

    Ordering: verified candidates first, then by soft score — the cap must drop
    the weakest traces, never the verified ones. Duplicates (same grid) are
    collapsed but their traces' sources noted, so the judge still sees natural
    multiplicity through repeated solution entries only up to the cap.
    """
    usable = [
        c for c in candidates
        if test_idx < len(c.grids) and c.grids[test_idx] is not None
    ]
    usable.sort(key=lambda c: (not c.verified, -c.soft, c.source, c.iteration))

    entries: list[dict] = []
    seen: set[tuple] = set()
    for c in usable:
        g = c.grids[test_idx]
        k = grid_key(g)
        if k in seen:
            continue
        seen.add(k)
        entries.append({"content": c.trace or c.code or "", "grid": g, "source": c.source})
        if len(entries) >= max_entries:
            break
    return entries


def parse_judge_response(text: str) -> list[Grid]:
    """Land's run_duo_pick_judge parsing: all CSV blocks, last two distinct."""
    grids = [g for g in (normalize_grid(b) for b in extract_all_grids(text)) if g]
    unique = distinct_grids(grids)
    return unique[-2:] if len(unique) >= 2 else unique


async def run_council(
    task_id: str,
    train_in: list[Grid],
    train_out: list[Grid],
    test_input: Grid,
    entries: list[dict],
    client: LLMClient,
    n_judges: int = 3,
    temperature: float = 1.0,
    reasoning_effort: str = "medium",
    max_tokens: int = 4096,
    request_timeout: float = 600.0,
    ledger: Optional[TokenLedger] = None,
    deadline: Optional[Deadline] = None,
    trace_cap_chars: int = 2500,
) -> list[JudgePick]:
    """Run n_judges samples over the SAME prompt (prefix cache => one prefill)."""
    if not entries:
        return []
    prompt = build_judge_prompt(
        train_in, train_out, test_input, entries,
        total_attempts=len(entries), trace_cap_chars=trace_cap_chars,
    )
    cache_key = f"judge:{task_id}:{grid_key(test_input)[:4]}"

    async def one(idx: int) -> JudgePick:
        if deadline is not None and deadline.level() >= 4:
            return JudgePick(error="deadline")
        resp = await client.complete(
            prompt,
            tag="judge",
            temperature=temperature,
            max_tokens=max_tokens,
            reasoning_effort=reasoning_effort,
            timeout=request_timeout,
            cache_key=cache_key,
        )
        if not resp.ok:
            return JudgePick(error=resp.error)
        grids = parse_judge_response(resp.text)
        return JudgePick(grids=grids, prompt_tokens=resp.prompt_tokens,
                          completion_tokens=resp.completion_tokens)

    results = await asyncio.gather(*[one(i) for i in range(n_judges)], return_exceptions=True)
    picks: list[JudgePick] = []
    for r in results:
        if isinstance(r, JudgePick):
            picks.append(r)
        else:
            picks.append(JudgePick(error=str(r)))
    return picks


def aggregate_picks(picks: list[JudgePick]) -> list[ScoreEntry]:
    """2 points for each judge's first choice, 1 point for each judge's second
    choice, summed per distinct output grid (Land's selection_advanced scoreboard).
    Grids not produced by any candidate program are 'synthesized'."""
    scoreboard: dict[tuple, ScoreEntry] = {}
    for run_idx, pick in enumerate(picks):
        for slot, grid in enumerate(pick.grids[:2]):
            points = 2 if slot == 0 else 1
            k = grid_key(grid)
            if k not in scoreboard:
                scoreboard[k] = ScoreEntry(grid=grid, origin="candidate", matched=True)
            scoreboard[k].points += points
            scoreboard[k].voters.append(run_idx)
    ranked = sorted(scoreboard.values(), key=lambda e: (-e.points,))
    return ranked


def mark_synthesized(entries_list: list[ScoreEntry], candidate_keys: set[tuple]) -> None:
    for e in entries_list:
        if grid_key(e.grid) not in candidate_keys:
            e.origin = "synthesized"
            e.matched = False


In [ ]:
%%writefile /kaggle/working/harness/selection.py
"""pass@2-aware attempt selection (HYPOTHESIS.md §5).

pass@2 = P(A1 correct) + P(A1 wrong) * P(A2 correct | A1 wrong), so attempt_2
must maximise CONDITIONAL COVERAGE, not raw rank. The decorrelation axis is the
output shape, checked against the train-implied shape prior S* (measured: the
train shape relation generalises to the test output in >=98% of eval tasks, so
shape errors are systematic and correlate across candidates).

Rules, in order:
  1. score every distinct grid (judge points + verified bonus + vote mass);
  2. attempt_1 = argmax score;
  3. shape-repair hedge: if S* known and shape(attempt_1) != S*, attempt_2 =
     best-scoring grid WITH shape S*;
  4. else attempt_2 = best-scoring different grid, tie-broken toward a different
     source channel and larger content distance (source decorrelation);
  5. supported-alternative swap: if the top-2 share shape AND source, and a
     different-shape grid with support (votes >= 2 or judge points >= 1) exists,
     swap it into attempt_2 (ambiguous-shape hedge, support-gated);
  6. hard guarantees: attempt_2 != attempt_1 as grids; missing slots fall back
     to identity(test_input), then the modal-train-output-colour grid.
"""

from __future__ import annotations

import math
from typing import Optional

from harness.grids import (
    grid_key,
    modal_color,
    normalized_distance,
    shape_of,
)
from harness.types import Candidate, Grid, GridStat, ScoreEntry

VERIFIED_BONUS_JUDGE = 2.0  # equal to one judge first-choice: a single noisy
# judge sample cannot displace a verified grid, a 2/3 majority (4+ pts) can.
VERIFIED_BONUS_NOJUDGE = 3.0
VOTE_WEIGHT_JUDGE = 0.25
VOTE_WEIGHT_NOJUDGE = 1.0
SOFT_WEIGHT_NOJUDGE = 0.5


def build_grid_stats(
    candidates: list[Candidate],
    test_idx: int,
    scoreboard: Optional[list[ScoreEntry]] = None,
) -> list[GridStat]:
    stats: dict[tuple, GridStat] = {}

    def get(grid: Grid) -> GridStat:
        k = grid_key(grid)
        if k not in stats:
            stats[k] = GridStat(grid=grid)
        return stats[k]

    for c in candidates:
        if test_idx < len(c.grids) and c.grids[test_idx] is not None:
            st = get(c.grids[test_idx])
            st.votes += 1
            st.verified_votes += int(c.verified)
            st.best_soft = max(st.best_soft, c.soft)
            st.sources.add(c.source)

    for e in scoreboard or []:
        st = get(e.grid)
        st.judge_points = max(st.judge_points, e.points)
        if e.origin == "synthesized":
            st.synthesized = True

    return list(stats.values())


def score_stat(st: GridStat, judge_on: bool) -> float:
    if judge_on:
        return st.judge_points + VERIFIED_BONUS_JUDGE * st.verified_votes + \
            VOTE_WEIGHT_JUDGE * math.log1p(st.votes)
    return VERIFIED_BONUS_NOJUDGE * st.verified_votes + VOTE_WEIGHT_NOJUDGE * math.log1p(st.votes) + \
        SOFT_WEIGHT_NOJUDGE * st.best_soft


def _rank(st: GridStat, judge_on: bool):
    """Deterministic ordering key: score, then verified, then votes, then soft,
    then grid key for total determinism."""
    return (
        -score_stat(st, judge_on),
        -st.verified_votes,
        -st.votes,
        -st.best_soft,
        str(grid_key(st.grid)),
    )


def early_accept(stats: list[GridStat]) -> Optional[GridStat]:
    """Skip the judge when >= 2 independent verified votes land on ONE grid and
    no other verified grid competes (verification precision ~97%, Land)."""
    verified = [st for st in stats if st.verified_votes > 0]
    if len(verified) != 1:
        return None
    top = verified[0]
    if top.verified_votes >= 2:
        return top
    return None


def select_attempts(
    stats: list[GridStat],
    test_input: Grid,
    train_out: list[Grid],
    suggested_shape: Optional[tuple[int, int]],
    judge_on: bool,
    diversity_on: bool = True,
) -> tuple[Grid, Grid]:
    """Return (attempt_1, attempt_2) — never equal, never empty."""
    fallbacks = _fallback_grids(test_input, train_out, suggested_shape)

    if not stats:
        return fallbacks[0], fallbacks[1]

    ranked = sorted(stats, key=lambda st: _rank(st, judge_on))
    a1 = ranked[0].grid

    if len(ranked) == 1:
        alt = next((f for f in fallbacks if grid_key(f) != grid_key(a1)), fallbacks[0])
        return a1, alt

    if not diversity_on:
        # naive top-2 (ablation baseline)
        return a1, ranked[1].grid

    a1_shape = shape_of(a1)
    rest = ranked[1:]

    # Rule 3: shape-repair hedge.
    if suggested_shape is not None and a1_shape != suggested_shape:
        matching = [st for st in rest if shape_of(st.grid) == suggested_shape]
        if matching:
            return a1, matching[0].grid

    # Rule 4: best remaining, tie-broken toward a different source and larger
    # content distance from attempt_1.
    def decorrelation_key(st: GridStat):
        base = _rank(st, judge_on)
        same_source = 0 if st.sources - (set(ranked[0].sources)) else 1
        dist = normalized_distance(st.grid, a1)
        # within near-ties (same integer score bucket) prefer different source
        # and larger distance; _rank already dominates.
        return (base[0], same_source, -dist) + base[1:]

    a2_stat = min(rest, key=decorrelation_key)

    # Rule 5: supported-alternative swap when top-2 are fully correlated
    # (same shape AND same source pool) and a corroborated different-shape
    # hypothesis exists.
    if (
        suggested_shape is None
        and shape_of(a2_stat.grid) == a1_shape
        and a2_stat.sources
        and a2_stat.sources <= set(ranked[0].sources)
    ):
        alternatives = [
            st for st in rest
            if shape_of(st.grid) != a1_shape
            and (st.votes >= 2 or st.judge_points >= 1 or st.verified_votes >= 1)
        ]
        if alternatives:
            a2_stat = min(alternatives, key=lambda st: _rank(st, judge_on))

    return a1, a2_stat.grid


def _fallback_grids(
    test_input: Grid, train_out: list[Grid], suggested_shape: Optional[tuple[int, int]]
) -> list[Grid]:
    identity = [list(row) for row in test_input]
    color = modal_color(train_out)
    shape = suggested_shape or shape_of(test_input) or (1, 1)
    filled = [[color] * shape[1] for _ in range(shape[0])]
    if grid_key(filled) == grid_key(identity):
        filled = [[(color + 1) % 10] * shape[1] for _ in range(shape[0])]
    return [identity, filled]


In [ ]:
%%writefile /kaggle/working/harness/submission.py
"""Cumulative submission writer + schema validation + watchdog finalization.

Kaggle contract (competition-brief.md):
  - every task_id in the test challenges MUST appear;
  - value = list with one {"attempt_1", "attempt_2"} dict per test input, in
    test-input order;
  - grids are rectangular int matrices; both attempts always present.
Written cumulatively after every task (Poetiq main.py pattern) and atomically
(tmp + os.replace) so a hard kill can never leave a truncated file.
"""

from __future__ import annotations

import json
import os
import tempfile
import threading
from typing import Callable, Optional

from harness.grids import modal_color
from harness.types import Grid


def fallback_attempts(test_inputs: list[Grid], train_out: list[Grid]) -> list[dict]:
    """Schema-valid placeholder attempts for an unsolved task: identity, then
    the modal train-output colour fill. Never empty."""
    color = modal_color(train_out) if train_out else 0
    out = []
    for tin in test_inputs:
        grid = tin if tin and tin[0] else [[0]]
        identity = [list(row) for row in grid]
        alt = [[(color + 1) % 10] * len(row) for row in grid]
        if alt == identity:
            alt = [[color] * len(row) for row in grid]
        out.append({"attempt_1": identity, "attempt_2": alt})
    return out


class SubmissionWriter:
    def __init__(self, path: str, challenges: dict):
        self.path = path
        self.challenges = challenges  # task_id -> {"train": [...], "test": [...]}
        self.submission: dict[str, list[dict]] = {}
        self._lock = threading.Lock()

    def update(self, task_id: str, attempts: list[dict]) -> None:
        with self._lock:
            self.submission[task_id] = attempts
            self._write_locked()

    def write(self) -> None:
        with self._lock:
            self._write_locked()

    def _write_locked(self) -> None:
        """Atomic write: tmp file in the same directory + os.replace.
        Safe against concurrent watchdog writes (lock) and hard kills (atomic)."""
        d = os.path.dirname(os.path.abspath(self.path)) or "."
        os.makedirs(d, exist_ok=True)
        fd, tmp = tempfile.mkstemp(dir=d, suffix=".tmp")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as f:
                json.dump(self.submission, f)
            os.replace(tmp, self.path)
        except BaseException:
            try:
                os.unlink(tmp)
            except OSError:
                pass
            raise

    def finalize(self, solver_fn: Optional[Callable[[str], Optional[list[dict]]]] = None) -> dict:
        """Fill every missing task (optionally via solver_fn) with schema-valid
        fallbacks, write once more, and return validation errors (should be [])."""
        with self._lock:
            for task_id, task in self.challenges.items():
                if task_id in self.submission and self.submission[task_id]:
                    continue
                attempts = None
                if solver_fn is not None:
                    try:
                        attempts = solver_fn(task_id)
                    except Exception:  # noqa: BLE001
                        attempts = None
                if not attempts:
                    test_inputs = [ex["input"] for ex in task.get("test", [])]
                    train_out = [ex["output"] for ex in task.get("train", [])]
                    attempts = fallback_attempts(test_inputs, train_out)
                self.submission[task_id] = attempts
            self._write_locked()
            return {"errors": validate_submission(self.submission, self.challenges)}


def validate_submission(submission: dict, challenges: dict) -> list[str]:
    """Full schema check. Returns a list of human-readable errors ([] = valid)."""
    errors: list[str] = []
    if not isinstance(submission, dict):
        return ["submission is not a dict"]
    for task_id, task in challenges.items():
        if task_id not in submission:
            errors.append(f"{task_id}: missing from submission")
            continue
        entries = submission[task_id]
        n_test = len(task.get("test", []))
        if not isinstance(entries, list):
            errors.append(f"{task_id}: value is not a list")
            continue
        if len(entries) != n_test:
            errors.append(f"{task_id}: expected {n_test} entries, got {len(entries)}")
            continue
        for j, entry in enumerate(entries):
            if not isinstance(entry, dict):
                errors.append(f"{task_id}[{j}]: entry is not a dict")
                continue
            for key in ("attempt_1", "attempt_2"):
                if key not in entry:
                    errors.append(f"{task_id}[{j}]: missing {key}")
                    continue
                if not _valid_grid(entry[key]):
                    errors.append(f"{task_id}[{j}]: {key} is not a rectangular int grid")
    for task_id in submission:
        if task_id not in challenges:
            errors.append(f"{task_id}: present in submission but not in challenges")
    return errors


def _valid_grid(g) -> bool:
    if not isinstance(g, list) or not g:
        return False
    width = None
    for row in g:
        if not isinstance(row, list) or not row:
            return False
        if width is None:
            width = len(row)
        elif len(row) != width:
            return False
        for c in row:
            if not isinstance(c, int) or isinstance(c, bool):
                return False
    return True


In [ ]:
%%writefile /kaggle/working/harness/config.py
"""Pipeline configuration + the deadline-driven degradation ladder (HYPOTHESIS.md §4.4)."""

from __future__ import annotations

from dataclasses import dataclass, replace
from typing import Optional

from harness.experts import ExpertConfig


@dataclass
class PipelineConfig:
    # experts
    experts: int = 2  # 0 = text-only, 1 = Expert A only, 2 = A + B
    iters_a: int = 8
    iters_b: int = 4
    pool_size: int = 5
    seed: int = 0
    solver_temperature: float = 1.0
    effort_solver: str = "low"
    max_tokens_solver: int = 2048
    sandbox_timeout: float = 5.0
    # text channel
    text_candidates: int = 3
    effort_text: str = "low"
    # judge
    use_judge: bool = True
    judges: int = 3
    effort_judge: str = "medium"
    judge_max_entries: int = 12
    judge_trace_cap_chars: int = 2500
    max_tokens_judge: int = 4096
    # selection
    diversity: bool = True
    # budget / time
    token_budget_per_task: int = 120_000  # GPU-effective
    total_time_s: float = 41_700.0
    max_task_seconds: float = 900.0
    task_concurrency: int = 2
    # client
    model: str = "qwen3.8-27b"
    base_url: str = "http://127.0.0.1:8000/v1"

    def expert_configs(self) -> list[ExpertConfig]:
        cfgs: list[ExpertConfig] = []
        if self.experts >= 1:
            cfgs.append(
                ExpertConfig(
                    name="expert_a",
                    variant="A",
                    max_iterations=self.iters_a,
                    # disjoint seed ranges (Poetiq solve_parallel_coding):
                    # each expert's iterations use seed + it, so ranges of
                    # max_iterations guarantee no seed collision.
                    seed=self.seed,
                    temperature=self.solver_temperature,
                    reasoning_effort=self.effort_solver,
                    max_tokens=self.max_tokens_solver,
                    pool_size=self.pool_size,
                    sandbox_timeout=self.sandbox_timeout,
                )
            )
        if self.experts >= 2:
            cfgs.append(
                ExpertConfig(
                    name="expert_b",
                    variant="B",
                    max_iterations=self.iters_b,
                    seed=self.seed + max(self.iters_a, 1) * 1000,
                    temperature=self.solver_temperature,
                    reasoning_effort=self.effort_solver,
                    max_tokens=self.max_tokens_solver,
                    pool_size=self.pool_size,
                    sandbox_timeout=self.sandbox_timeout,
                )
            )
        return cfgs


def degraded(cfg: PipelineConfig, level: int) -> PipelineConfig:
    """Degradation ladder driven by Deadline.level():
      L1 @60%: iters A 8->5, text 3->1
      L2 @75%: iters ->3, no text, 1 judge
      L3 @88%: single-shot experts, no judge, consensus-only
      L4 @93%: handled by the runner (stop solving, finalize).
    """
    if level <= 0:
        return cfg
    if level == 1:
        return replace(cfg, iters_a=min(cfg.iters_a, 5), iters_b=min(cfg.iters_b, 3),
                       text_candidates=min(cfg.text_candidates, 1))
    if level == 2:
        return replace(cfg, iters_a=min(cfg.iters_a, 3), iters_b=min(cfg.iters_b, 2),
                       text_candidates=0, judges=1)
    return replace(cfg, iters_a=min(cfg.iters_a, 1), iters_b=min(cfg.iters_b, 1),
                   text_candidates=0, use_judge=False, judges=1)


In [ ]:
%%writefile /kaggle/working/harness/runner.py
"""Task + run orchestration.

solve_task: experts and text channel concurrently -> per-test-input candidate
stats -> early-accept or judge council -> pass@2-aware attempt selection.

TaskRunner: deadline-aware loop over a challenge set with cumulative submission
writes, per-task scoring against ground truth (when available), a watchdog
finalization hook, and the degradation ladder.
"""

from __future__ import annotations

import asyncio
import threading
import time
from typing import Optional

from harness.budget import Deadline, TokenLedger
from harness.config import PipelineConfig, degraded
from harness.experts import CodeExpert, TaskData, TextChannel
from harness.grids import grid_key, suggest_output_shape
from harness.judge import aggregate_picks, build_judge_entries, mark_synthesized, run_council
from harness.llm import LLMClient
from harness.selection import build_grid_stats, early_accept, score_stat, select_attempts
from harness.submission import SubmissionWriter
from harness.types import Candidate, Grid


async def solve_task(
    task_id: str,
    task: dict,
    client: LLMClient,
    cfg: PipelineConfig,
    ledger: TokenLedger,
    deadline: Optional[Deadline] = None,
    task_deadline_s: Optional[float] = None,
) -> tuple[list[dict], dict]:
    """Solve one task. Returns (attempts, meta) where attempts is the
    submission-schema list of {"attempt_1", "attempt_2"} per test input."""
    train_in: list[Grid] = [ex["input"] for ex in task["train"]]
    train_out: list[Grid] = [ex["output"] for ex in task["train"]]
    test_in: list[Grid] = [ex["input"] for ex in task["test"]]
    td = TaskData(task_id=task_id, train_in=train_in, train_out=train_out, test_in=test_in)

    level = deadline.level() if deadline is not None else 0
    ecfg = degraded(cfg, level)

    ledger.begin_task(task_id)
    meta: dict = {"task_id": task_id, "level": level, "judged": [], "verified": False}

    # --- candidate generation (all channels concurrently) -------------------
    jobs = []
    for expert_cfg in ecfg.expert_configs():
        if expert_cfg.max_iterations <= 0:
            continue
        jobs.append(CodeExpert(expert_cfg).run(td, client, ledger, deadline, task_deadline_s))
    if ecfg.text_candidates > 0 and ecfg.experts >= 0:
        jobs.append(
            TextChannel(k=ecfg.text_candidates, reasoning_effort=ecfg.effort_text).run(
                td, client, ledger, deadline
            )
        )
    results = await asyncio.gather(*jobs, return_exceptions=True)

    candidates: list[Candidate] = []
    channel_errors = 0
    for r in results:
        if isinstance(r, BaseException):
            channel_errors += 1
            continue
        candidates.extend(r)
    meta["candidates"] = len(candidates)
    meta["channel_errors"] = channel_errors
    meta["verified"] = any(c.verified for c in candidates)

    # --- per-test-input selection -------------------------------------------
    attempts: list[dict] = []
    for j, tin in enumerate(test_in):
        suggested = suggest_output_shape(train_in, train_out, tin)
        meta.setdefault("suggested_shapes", []).append(suggested)

        scoreboard = None
        judge_on = False
        stats_pre = build_grid_stats(candidates, j, None)
        accepted = early_accept(stats_pre)

        if accepted is None and ecfg.use_judge and ecfg.judges > 0:
            entries = build_judge_entries(candidates, j, max_entries=ecfg.judge_max_entries)
            if entries:  # even one grid is worth a judge pass (synthesis can rescue it)
                picks = await run_council(
                    task_id, train_in, train_out, tin, entries, client,
                    n_judges=ecfg.judges,
                    reasoning_effort=ecfg.effort_judge,
                    max_tokens=ecfg.max_tokens_judge,
                    ledger=ledger,
                    deadline=deadline,
                    trace_cap_chars=ecfg.judge_trace_cap_chars,
                )
                ok_picks = [p for p in picks if p.grids]
                if ok_picks:
                    scoreboard = aggregate_picks(picks)
                    cand_keys = {grid_key(e["grid"]) for e in entries}
                    mark_synthesized(scoreboard, cand_keys)
                    judge_on = True
                    meta["judged"].append(j)

        stats = build_grid_stats(candidates, j, scoreboard)
        a1, a2 = select_attempts(
            stats, tin, train_out, suggested,
            judge_on=judge_on, diversity_on=ecfg.diversity,
        )
        attempts.append({"attempt_1": a1, "attempt_2": a2})

    totals = ledger.end_task()
    meta["tokens"] = totals
    return attempts, meta


def score_attempts(attempts: list[dict], gt_outputs: list[Grid]) -> tuple[int, int, int]:
    """(pass@2 hits, pass@1 hits, n_outputs) for one task; a hit requires an
    exact grid match (dims + every cell) by either / the first attempt."""
    hits2 = hits1 = 0
    for j, gt in enumerate(gt_outputs):
        if j >= len(attempts):
            continue
        a1 = attempts[j].get("attempt_1")
        a2 = attempts[j].get("attempt_2")
        if a1 == gt:
            hits1 += 1
        if a1 == gt or a2 == gt:
            hits2 += 1
    return hits2, hits1, len(gt_outputs)


class TaskRunner:
    def __init__(
        self,
        challenges: dict,
        client: LLMClient,
        cfg: PipelineConfig,
        out_path: str,
        solutions: Optional[dict] = None,
        deadline: Optional[Deadline] = None,
    ):
        self.challenges = challenges
        self.client = client
        self.cfg = cfg
        self.solutions = solutions
        self.deadline = deadline or Deadline(cfg.total_time_s)
        self.ledger = TokenLedger(per_task_effective_cap=cfg.token_budget_per_task)
        # make the client account into this runner's ledger unless it has one
        if getattr(client, "ledger", None) is None:
            try:
                client.ledger = self.ledger
            except AttributeError:
                pass
        self.writer = SubmissionWriter(out_path, challenges)
        self.metas: dict[str, dict] = {}
        self.stop_requested = False
        self._lock = asyncio.Lock()

    def request_stop(self) -> None:
        """Watchdog hook: stop scheduling new tasks and finalize."""
        self.stop_requested = True

    def start_watchdog(self, finalize_before_s: float = 60.0) -> threading.Thread:
        """Daemon thread that force-finalizes the submission at
        deadline - finalize_before_s, whatever the solver is doing (lb3389's
        600s-buffer pattern). Safe to call multiple times."""
        def _watch():
            wait = max(0.0, self.deadline.remaining() - finalize_before_s)
            time.sleep(wait)
            self.request_stop()
            try:
                self.writer.finalize()
            except Exception:  # noqa: BLE001 — last resort: never raise in a watchdog
                pass

        t = threading.Thread(target=_watch, daemon=True, name="submission-watchdog")
        t.start()
        self._watchdog = t
        return t

    async def run(self, limit: Optional[int] = None, task_ids: Optional[list[str]] = None) -> dict:
        ids = task_ids if task_ids is not None else sorted(self.challenges.keys())
        if limit is not None:
            ids = ids[:limit]
        sem = asyncio.Semaphore(max(1, self.cfg.task_concurrency))
        start = time.monotonic()
        results: dict[str, dict] = {}
        done = {"n": 0}

        async def one(task_id: str) -> None:
            async with sem:
                if self.stop_requested or self.deadline.level() >= 4:
                    done["n"] += 1
                    return
                remaining = max(1, len(ids) - done["n"])
                t_soft = self.deadline.task_deadline(remaining, self.cfg.max_task_seconds)
                t0 = time.monotonic()
                try:
                    attempts, meta = await solve_task(
                        task_id, self.challenges[task_id], self.client, self.cfg,
                        self.ledger, self.deadline, task_deadline_s=t_soft,
                    )
                except Exception as e:  # noqa: BLE001 — never lose the run
                    attempts, meta = [], {"task_id": task_id, "error": f"{type(e).__name__}: {e}"}
                meta["wall_s"] = round(time.monotonic() - t0, 1)
                done["n"] += 1
                async with self._lock:
                    self.metas[task_id] = meta
                    if attempts:
                        self.writer.update(task_id, attempts)
                    if self.solutions is not None and task_id in self.solutions:
                        gt = self.solutions[task_id]
                        h2, h1, n = score_attempts(attempts, gt) if attempts else (0, 0, len(gt))
                        results[task_id] = {"hits2": h2, "hits1": h1, "n": n}
                        meta["hits2"], meta["hits1"], meta["n_outputs"] = h2, h1, n

        await asyncio.gather(*(one(tid) for tid in ids))

        # cumulative writer holds solved tasks; fill the rest with fallbacks
        fin = self.writer.finalize()
        summary = self._summarize(ids, results, start)
        summary["validation_errors"] = fin["errors"]
        return summary

    def _summarize(self, ids: list[str], results: dict[str, dict], start: float) -> dict:
        usage = self.ledger.snapshot()
        n_out = sum(r["n"] for r in results.values())
        h2 = sum(r["hits2"] for r in results.values())
        h1 = sum(r["hits1"] for r in results.values())
        metas = list(self.metas.values())
        return {
            "tasks_run": len(ids),
            "tasks_scored": len(results),
            "outputs_scored": n_out,
            "pass2": (h2 / n_out) if n_out else None,  # competition metric
            "pass1": (h1 / n_out) if n_out else None,
            "tasks_fully_solved_pass2": sum(
                1 for r in results.values() if r["n"] and r["hits2"] == r["n"]
            ),
            "verified_tasks": sum(1 for m in metas if m.get("verified")),
            "judged_test_inputs": sum(len(m.get("judged", [])) for m in metas),
            "channel_errors": sum(m.get("channel_errors", 0) for m in metas),
            "wall_s": round(time.monotonic() - start, 1),
            "tokens": usage,
        }


In [ ]:
%%writefile /kaggle/working/harness/kaggle_main.py
"""Kaggle rerun orchestration: locate the attached model, launch vLLM (TP=4,
offline), run the pipeline over the test challenges with the watchdog, and
guarantee a schema-valid /kaggle/working/submission.json.

Everything degrades to a fallback-only submission (never lose the submission):
no GPU -> fallback; vLLM missing -> wheelhouse install -> fallback; server never
becomes healthy -> fallback.
"""

from __future__ import annotations

import asyncio
import json
import os
import shutil
import subprocess
import sys
import time
from typing import Optional

COMPETITION_DIR = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
WORKING_DIR = "/kaggle/working"
MODEL_CANDIDATES = (
    "/kaggle/input/qwen3-8-27b-fp8",
    "/kaggle/input/qwen3-8-27b-fp8-hf-snapshot",
    "/kaggle/input/qwen3-8-27b-bf16",
    "/kaggle/input/qwen3-8-27b",
)
VLLM_ARGS = [
    "--served-model-name", "qwen3.8-27b",
    "--tensor-parallel-size", "4",
    "--max-model-len", "49152",
    "--gpu-memory-utilization", "0.90",
    "--reasoning-parser", "qwen3",
    "--enable-auto-tool-choice", "--tool-call-parser", "qwen3_coder",
    "--enable-prefix-caching", "--mamba-cache-mode", "align",
    "--max-num-seqs", "16",
    "--port", "8000",
]


def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)


def find_model_dir() -> Optional[str]:
    for cand in MODEL_CANDIDATES:
        if os.path.isdir(cand):
            for root, _dirs, files in os.walk(cand):
                if any(f.endswith(".safetensors") for f in files) and any(
                    f == "config.json" for f in files
                ):
                    return root
    return None


def have_vllm() -> bool:
    try:
        import vllm  # noqa: F401

        return True
    except Exception:
        return False


def install_vllm_from_wheelhouse() -> bool:
    """Offline rerun: pip-install from an attached wheelhouse dataset if present."""
    root = "/kaggle/input"
    if not os.path.isdir(root):
        return False
    wheels = []
    for name in sorted(os.listdir(root)):
        d = os.path.join(root, name)
        try:
            if os.path.isdir(d) and any(f.endswith(".whl") for f in os.listdir(d)):
                wheels.append(d)
        except OSError:
            continue
    for d in wheels:
        log(f"trying wheelhouse {d}")
        r = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--no-index", "--find-links", d, "vllm"],
            capture_output=True, text=True,
        )
        if r.returncode == 0 and have_vllm():
            return True
    return False


def launch_vllm(model_dir: str, log_path: str) -> Optional[subprocess.Popen]:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", model_dir] + VLLM_ARGS
    # env pins: never let reasoning_effort default (xhigh) matter — we pass it
    # per request; disable telemetry noise.
    env = dict(os.environ)
    env.setdefault("VLLM_LOGGING_LEVEL", "WARNING")
    log(f"launching: {' '.join(cmd)}")
    with open(log_path, "w", encoding="utf-8") as lf:
        return subprocess.Popen(cmd, stdout=lf, stderr=subprocess.STDOUT, env=env)


def wait_for_server(base_url: str, timeout_s: float, proc: Optional[subprocess.Popen]) -> bool:
    import urllib.request

    end = time.monotonic() + timeout_s
    while time.monotonic() < end:
        if proc is not None and proc.poll() is not None:
            log(f"vLLM process exited with code {proc.returncode}")
            return False
        try:
            with urllib.request.urlopen(base_url + "/models", timeout=5) as r:
                if r.status == 200:
                    log("vLLM server healthy")
                    return True
        except Exception:
            time.sleep(10)
    return False


def load_challenges() -> tuple[dict, Optional[dict]]:
    rerun = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
    if rerun:
        path = os.path.join(COMPETITION_DIR, "arc-agi_test_challenges.json")
        solutions = None
    else:
        path = os.path.join(COMPETITION_DIR, "arc-agi_evaluation_challenges.json")
        solutions = None
        sol_path = os.path.join(COMPETITION_DIR, "arc-agi_evaluation_solutions.json")
        if os.path.exists(sol_path):
            with open(sol_path, encoding="utf-8") as f:
                solutions = json.load(f)
    with open(path, encoding="utf-8") as f:
        challenges = json.load(f)
    log(f"loaded {len(challenges)} tasks from {path} (rerun={rerun})")
    return challenges, solutions


def main(total_time_s: Optional[float] = None) -> dict:
    from harness.budget import Deadline
    from harness.config import PipelineConfig
    from harness.llm import OpenAICompatClient
    from harness.runner import TaskRunner

    out_path = os.path.join(WORKING_DIR, "submission.json")
    challenges, solutions = load_challenges()

    # 12h - 600s buffer (lb3389 pattern); minus the serving setup measured below.
    setup_start = time.monotonic()
    cfg = PipelineConfig()
    deadline = Deadline(total_time_s if total_time_s is not None else cfg.total_time_s)

    writer_ready = False
    proc = None
    base_url = "http://127.0.0.1:8000/v1"
    model_dir = find_model_dir()
    gpu = shutil.which("nvidia-smi") is not None and subprocess.run(
        ["nvidia-smi"], capture_output=True
    ).returncode == 0

    if not gpu:
        log("NO GPU visible — fallback-only submission mode")
    elif model_dir is None:
        log("model dataset not found — fallback-only submission mode")
    else:
        if not have_vllm():
            log("vllm not importable; trying offline wheelhouse")
            if not install_vllm_from_wheelhouse():
                log("vLLM unavailable — fallback-only submission mode")
                model_dir = None
        if model_dir is not None:
            proc = launch_vllm(model_dir, os.path.join(WORKING_DIR, "vllm.log"))
            load_budget = 1800.0  # generous: FP8 27GB load + graph capture
            if not wait_for_server(base_url, load_budget, proc):
                log("vLLM never became healthy — fallback-only submission mode")
                if proc is not None:
                    proc.kill()
                proc = None
            else:
                # shrink the usable clock by the time setup actually took
                spent = time.monotonic() - setup_start
                deadline = Deadline(max(600.0, (total_time_s or cfg.total_time_s) - spent))
                writer_ready = True

    runner = TaskRunner(
        challenges,
        client=OpenAICompatClient(base_url=base_url, model=cfg.model),
        cfg=cfg,
        out_path=out_path,
        solutions=solutions,
        deadline=deadline,
    )
    runner.start_watchdog(finalize_before_s=60.0)

    if not writer_ready:
        summary = runner.writer.finalize()
        log(f"fallback submission written; validation errors: {summary['errors']}")
        return {"mode": "fallback", **summary}

    summary = asyncio.run(runner.run())
    log("summary: " + json.dumps({k: v for k, v in summary.items() if k != "tokens"}))
    log("tokens: " + json.dumps(summary.get("tokens", {})))
    if proc is not None:
        proc.terminate()
    return {"mode": "full", **summary}


if __name__ == "__main__":
    total = float(sys.argv[1]) if len(sys.argv) > 1 else None
    main(total)


In [ ]:
# --- sanity: import the harness written above ---
import harness
from harness import kaggle_main
from harness.config import PipelineConfig
from harness.submission import validate_submission
print("harness import OK:", harness.__file__)
model_dir = kaggle_main.find_model_dir()
print("model dir:", model_dir)
print("vllm importable:", kaggle_main.have_vllm())

In [ ]:
# --- main run: vLLM launch (TP=4, offline) + experts + judge + watchdog ---
# Blocks until the run finishes or the deadline ladder finalizes the submission.
# Falls back to a schema-valid placeholder submission when GPUs/model/vLLM are
# unavailable, so the kernel always produces /kaggle/working/submission.json.
from harness import kaggle_main

total_time_s = max(600.0, GLOBAL_END_TIME - time.time())
summary = kaggle_main.main(total_time_s=total_time_s)
print("mode:", summary.get("mode"))

In [ ]:
# --- final validation (always runs; guarantees the contract even if the cell
# above crashed): every task_id present, one {attempt_1, attempt_2} per test
# input in order, rectangular int grids, both attempts non-empty ---
import json, os
from harness.submission import SubmissionWriter, validate_submission

comp_dir = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
rerun_mode = bool(os.getenv("KAGGLE_IS_COMPETITION_RERUN"))
challenges_path = os.path.join(
    comp_dir,
    "arc-agi_test_challenges.json" if rerun_mode else "arc-agi_evaluation_challenges.json",
)
with open(challenges_path) as f:
    challenges = json.load(f)

sub_path = "/kaggle/working/submission.json"
if os.path.exists(sub_path):
    with open(sub_path) as f:
        sub = json.load(f)
else:
    sub = {}

writer = SubmissionWriter(sub_path, challenges)
writer.submission = {k: v for k, v in sub.items() if k in challenges and v}
res = writer.finalize()
print("validation errors:", res["errors"])
assert not res["errors"], res["errors"][:5]

# score against ground truth when available (interactive / eval-split runs)
sol_path = os.path.join(comp_dir, "arc-agi_evaluation_solutions.json")
if not rerun_mode and os.path.exists(sol_path):
    from harness.runner import score_attempts
    with open(sol_path) as f:
        solutions = json.load(f)
    hits = n = 0
    for tid, outs in solutions.items():
        if tid in challenges and tid in sub:
            h2, _h1, k = score_attempts(sub[tid], outs)
            hits += h2
            n += k
    print(f"local eval-split pass@2: {hits}/{n} = {hits / max(n, 1) * 100:.2f}%")
print("submission written:", sub_path, os.path.getsize(sub_path), "bytes")

## Local development & ablations (outside Kaggle)

The identical harness runs against any OpenAI-compatible endpoint or the deterministic MockLLM:

```bash
python -m pytest tests -q                          # 157 tests, no GPU needed
python scripts/eval_local.py --mock --limit 10     # pipeline dry run
python scripts/eval_local.py --base-url http://127.0.0.1:8000/v1   # real model

# ablations (each component's contribution is measurable):
python scripts/eval_local.py --no-judge            # judge on/off
python scripts/eval_local.py --no-diversity        # pass@2 selection on/off
python scripts/eval_local.py --experts 1 --pool-size 1   # chain-refinement (Land's failure mode)
python scripts/eval_local.py --experts 2 --pool-size 5   # population refinement (Poetiq)
```